In [3]:
!pip install lifelines

# Fix numpy/pyarrow conflict caused by mixed conda+pip environments
!pip install numpy --force-reinstall
!pip install pyarrow --force-reinstall

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
  Using cached numpy-2.4.4-cp311-cp311-win_amd64.whl.metadata (6.6 kB)
Using cached numpy-2.4.4-cp311-cp311-win_amd64.whl (12.6 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.4.4
    Uninstalling numpy-2.4.4:
      Successfully uninstalled numpy-2.4.4


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gensim 4.3.0 requires FuzzyTM>=0.4.0, which is not installed.
genai-core 2.7.1 requires numpy==1.26.4, but you have numpy 2.4.4 which is incompatible.
genai-core 2.7.1 requires openai<2.0.0,>=1.34.0, but you have openai 2.14.0 which is incompatible.
mlflow 3.9.0 requires pyarrow<23,>=4.0.0, but you have pyarrow 23.0.1 which is incompatible.
opencv-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 2.4.4 which is incompatible.
unstructured 0.16.25 requires numpy<2, but you have numpy 2.4.4 which is incompatible.
astropy 5.3.4 requires numpy<2,>=1.21, but you have numpy 2.4.4 which is incompati

Defaulting to user installation because normal site-packages is not writeable
  Using cached pyarrow-23.0.1-cp311-cp311-win_amd64.whl.metadata (3.1 kB)
Using cached pyarrow-23.0.1-cp311-cp311-win_amd64.whl (27.5 MB)
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 23.0.1
    Uninstalling pyarrow-23.0.1:
      Successfully uninstalled pyarrow-23.0.1


  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mlflow 3.9.0 requires pyarrow<23,>=4.0.0, but you have pyarrow 23.0.1 which is incompatible.
streamlit 1.30.0 requires cachetools<6,>=4.0, but you have cachetools 6.2.2 which is incompatible.
streamlit 1.30.0 requires numpy<2,>=1.19.3, but you have numpy 2.4.4 which is incompatible.
streamlit 1.30.0 requires packaging<24,>=16.8, but you have packaging 25.0 which is incompatible.
streamlit 1.30.0 requires pillow<11,>=7.1.0, but you have pillow 12.0.0 which is incompatible.
streamlit 1.30.0 requires protobuf<5,>=3.20, but you have protobuf 5.29.5 which is incompatible.
streamlit 1.30.0 requires rich<14,>=10.14.0, but you have rich 14.2.0 which is incompatible.
streamlit 1.30.0 requires tenacity<9,>=8.1.0, but you have tenacity 9.1.2 which is i

# Pricing Optimisation — Acceptance Probability & Expected Financial Loss

## Purpose
This notebook implements the **black-box model generation pipeline** used downstream by the pricing optimisation module.  
It produces four output CSV files consumed by the optimiser:

| Output file | Content | Model source |
|---|---|---|
| `df_acceptance_xgb_black_box.csv` | Acceptance probability per policy | XGBoost classifier |
| `df_exp_financial_loss_xgb_black_box.csv` | Expected financial loss per policy | XGBoost regressor |
| `df_acceptance_linear_model_black_box.csv` | Acceptance probability per policy | GLM (Logistic Regression) |
| `df_acceptance_gam_model_black_box.csv` | Acceptance probability per policy | GAM (LogisticGAM) |
| `df_exp_financial_loss_lner_black_box.csv` | Expected financial loss per policy | Linear  |
| 

Additionally, it serialises all trained models to `artifacts/` as pickle files for reuse (see Section 17).

## Workflow Overview
1. **Setup** — install dependencies  
2. **Data ingestion** — load and clean the 3m-1006 parquet dataset  
3. **Feature engineering** — rename columns, compute premium change ratio `U`  
4. **Outlier filtering** — apply per-column IQR / percentile bounds  
5. **Model training (cross-validation with OOF predictions)**  
   - XGBoost regressor: predicts current policy premium `Y`  
   - XGBoost classifier: predicts churn probability  
   - GLM (logistic regression): interpretable churn alternative  
   - GAM (LogisticGAM): semi-parametric churn alternative  
   - Linear regression: interpretable premium baseline  
6. **Output generation** — derive acceptance probability and save CSVs  
7. **Model serialisation** — save all four models to `artifacts/`

In [4]:
!pip install --upgrade scikit-learn optuna

Defaulting to user installation because normal site-packages is not writeable


## Section 2 — Data Ingestion & Preprocessing

Loads the anonymised 3-month policy dataset (`full_dataset_anonimized_processed_3m_1006.parquet`) and applies the following quality filters:

- **Missing upcoming renewal data** — rows with sentinel values for premium (`9999`), premium % change (`9`) or driver age (`100`) are dropped.
- **Duplicate churn rows** — only the first churn event per policy is retained; subsequent rows after churn are removed.
- **Mismatch policies** — policies where the *supposed* next-year premium differs from the *actual* premium by more than €40 (or the % change differs by more than 0.2) are excluded entirely to avoid noisy training signal.

The analysis is restricted to `year == 7` through the `df_sub` filter below.

In [5]:
import pandas as pd

def preprocess_3m_1006(parquet_path: str):
    """"
    #### Args
    - parquet_path: path to parquet file containing the 3m_1006 dataset

    #### Returns
    - df: preprocessed dataframe
    """
    df = pd.read_parquet(parquet_path)
    # make sure df is sorted by policy number first then year
    df = df.sort_values(['encoded_policy_number', 'year']).reset_index(drop=True)
    print(f"Original rows {len(df)}, Unique policies: {df['encoded_policy_number'].nunique()}")

    # get rid of rows with missing upcoming renewal data
    df = df[
        (df['premio_total_continuado_x_impute_GTE100_LTE1500_missing_9999'] != 9999) & # upcoming renewal premium
        (df['tx_pt_continuado_x_impute_inflation_this_renewal_6m_GTEminus1_LTE1_missing_9'] != 9) & # cur renewal % change
        (df['idade_condutor_x_impute_GTE18_LTE100_map_missing_100'] != 100) # age
    ]

    # remove churned rows that are not the first churned row of each policy
    is_first_policy_row = df["encoded_policy_number"] != df["encoded_policy_number"].shift(1)
    mask = (
        (df["is_churn"] == False) |
        (
            (df["is_churn"] == True) &
            (
                is_first_policy_row |
                (df["is_churn"].shift(1) == False)
            )
        )
    )
    df = df[mask]

    # ----- remove entire policies with mismatches (supposed increase vs actual increase) 
    # ----- over the thresholds
    # premium mismatch
    supposed = "premio_total_continuado_x_impute_GTE100_LTE1500_missing_9999"
    actual = "valor_vigor_pt_apol"

    temp = df[['encoded_policy_number', actual, supposed]].copy()
    temp['diff'] = (temp[actual] - temp[supposed].shift(1)).fillna(0).abs()

    same_policy_mask = temp['encoded_policy_number'] == temp['encoded_policy_number'].shift(1)
    temp = temp[same_policy_mask]
    mismatches = temp.groupby('encoded_policy_number')['diff'].max() > 40 # threshold

    bad_policies = set(mismatches.index[mismatches])
    
    # premium % change mismatch
    actual = "tx_pt_continuado_ant_x_impute_inflation_last_renewal_6m_GTEminus1_LTE1_missing_9"
    supposed = "tx_pt_continuado_x_impute_inflation_this_renewal_6m_GTEminus1_LTE1_missing_9"

    temp = df[['encoded_policy_number', actual, supposed]].copy()
    temp = temp[temp[actual] != 9.0]
    temp['diff'] = (temp[actual] - temp[supposed].shift(1)).fillna(0).abs()

    same_policy_mask = temp['encoded_policy_number'] == temp['encoded_policy_number'].shift(1)
    temp = temp[same_policy_mask]
    mismatches = temp.groupby('encoded_policy_number')['diff'].max() > 0.2 # threshold

    bad_policies |= set(mismatches.index[mismatches])

    # remove bad policies from df
    df = df[~df['encoded_policy_number'].isin(bad_policies)]
    print(f"Remaining rows {len(df)}, Remaining unique policies: {df['encoded_policy_number'].nunique()}")

    return df


## Section 1 — Setup & Dependencies

In [190]:
from preprocessing import *
import numpy as np
import pprint as pp
from matplotlib import pyplot as plt


In [7]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

%matplotlib inline


from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder



In [8]:
import sklearn

In [305]:
file_path = r"full_dataset_anonimized_processed_3m_1006.parquet"
df = preprocess_3m_1006(file_path)

Original rows 3370487, Unique policies: 1103634
Remaining rows 3075795, Remaining unique policies: 1034510


In [306]:
df.head()

,encoded_policy_number,idade_condutor_x_impute_GTE18_LTE100_map_missing_100,indice_bonus_malus,canal_distribuicao_h4_x_grouping_all_products,cod_estatistico_classe_risco,num_sin_tot,num_apolices_tot,ind_cliente_b_x_grouping_all_products,idade_construcao_veiculo_x_impute_GTE0_LTE50_missing_0,antiguidade_apolice_x_impute_GTE0_LTE30_missing_0,valor_vigor_pt_apol,tx_pt_continuado_ant_x_impute_inflation_last_renewal_6m_GTEminus1_LTE1_missing_9,premio_total_continuado_x_impute_GTE100_LTE1500_missing_9999,tx_pt_continuado_x_impute_inflation_this_renewal_6m_GTEminus1_LTE1_missing_9,year,is_churn
0,5,86.0,50.000,19,T4005040,0,1,0,17.0,17.0,174.4700,-0.108391,157.14,-0.111998,0,False
1,5,87.0,47.500,19,T4005040,0,1,0,18.0,18.0,157.1400,-0.111998,156.74,-0.011103,1,False
2,5,88.0,45.125,19,T4005040,0,1,0,19.0,19.0,156.7400,-0.011103,156.36,-0.009236,2,False
3,5,89.0,45.000,19,T4005040,0,1,0,20.0,20.0,155.9369,-0.011918,164.11,0.051139,3,False
4,5,90.0,45.000,19,T4005040,0,1,0,21.0,21.0,164.1100,0.048342,163.72,-0.007067,4,False


In [11]:
!pip install -U glum

Defaulting to user installation because normal site-packages is not writeable


In [230]:
df_sub.columns

Index(['encode_policy_number', 'X_age', 'X_bonus_malus_rating',
       'X_distr_channel', 'X_vehicle_type', 'ttm_claims', 'X_policy_count',
       'X_risk_code', 'X_vehicle_age', 'X_policy_tenure', 'X_policy_premium',
       'X_prev_renewal_perc', 'X_upcoming_premium', 'X_cur_renewal_perc',
       'year', 'is_churn'],
      dtype='object')

In [307]:
df_sub = df.loc[df.year==7]


## Section 3 — Feature Name Mapping

Raw column names in the parquet file use verbose Portuguese/encoded names.  
This section defines a mapping to clean `X_` prefixed names used throughout the modelling pipeline.

| Clean name | Meaning |
|---|---|
| `X_age` | Driver age (clamped 18–100) |
| `X_vehicle_age` | Vehicle age in years |
| `X_policy_tenure` | Policy seniority in years |
| `X_policy_count` | Number of policies held by this client |
| `X_risk_code` | Risk segment code (higher = lower risk) |
| `X_distr_channel` | Distribution channel group |
| `X_ttm_claims` | Total claims in trailing 12 months |
| `X_bonus_malus_rating` | Bonus-malus coefficient |
| `X_vehicle_type` | Vehicle statistical risk class |
| `X_policy_premium` | Current active premium (`Y` in regression) |
| `X_upcoming_premium` | Proposed renewal premium |
| `X_cur_renewal_perc` | % premium change at this renewal |
| `U` | Relative premium increase: `(upcoming - current) / current` |

In [308]:

age = 'idade_condutor_x_impute_GTE18_LTE100_map_missing_100'
vehicle_age = 'idade_construcao_veiculo_x_impute_GTE0_LTE50_missing_0' # 0 = new car
policy_tenure = 'antiguidade_apolice_x_impute_GTE0_LTE30_missing_0'
policy_count = 'num_apolices_tot'
risk_code = 'ind_cliente_b_x_grouping_all_products' # integers, larger = higher survival chance
distr_channel = 'canal_distribuicao_h4_x_grouping_all_products' # larger = higher obs survival in training
ttm_claims = 'num_sin_tot'
bonus_malus_rating = 'indice_bonus_malus'
vehicle_type = 'cod_estatistico_classe_risco'
policy_premium = 'valor_vigor_pt_apol'
upcoming_premium = 'premio_total_continuado_x_impute_GTE100_LTE1500_missing_9999'
prev_renewal_perc = "tx_pt_continuado_ant_x_impute_inflation_last_renewal_6m_GTEminus1_LTE1_missing_9"
cur_renewal_perc = 'tx_pt_continuado_x_impute_inflation_this_renewal_6m_GTEminus1_LTE1_missing_9' # % change of total premium
id_number = 'encode_policy_number'

# for selecting the features to build the model
covariate_cols = [age, vehicle_age, policy_count, risk_code, distr_channel, ttm_claims,bonus_malus_rating, vehicle_type,
                  policy_premium, upcoming_premium, cur_renewal_perc]
col_name_map = {
    id_number: 'encode_policy_number',
    age: 'X_age',
    vehicle_age: 'X_vehicle_age',
    policy_tenure: 'X_policy_tenure',
    policy_count: 'X_policy_count',
    risk_code: 'X_risk_code',
    distr_channel: 'X_distr_channel',
    ttm_claims: 'X_ttm_claims',
    bonus_malus_rating: 'X_bonus_malus_rating',
    vehicle_type: 'X_vehicle_type',
    policy_premium: 'X_policy_premium',
    upcoming_premium: 'X_upcoming_premium',
    prev_renewal_perc:"X_prev_renewal_perc",
    cur_renewal_perc: 'X_cur_renewal_perc'
}

In [309]:
df_sub = df_sub.rename(columns=col_name_map) 

In [310]:
df_sub.columns

Index(['encoded_policy_number', 'X_age', 'X_bonus_malus_rating',
       'X_distr_channel', 'X_vehicle_type', 'X_ttm_claims', 'X_policy_count',
       'X_risk_code', 'X_vehicle_age', 'X_policy_tenure', 'X_policy_premium',
       'X_prev_renewal_perc', 'X_upcoming_premium', 'X_cur_renewal_perc',
       'year', 'is_churn'],
      dtype='object')

In [245]:
# common function to filter out rows fro xgb regression and classification 

## Section 4 — Outlier Filtering: `filter_middle_advanced`

`filter_middle_advanced` applies **per-column** outlier filtering before model training.  
Three methods are supported:

- **`iqr`** — keeps rows within `[Q1 − k·IQR, Q3 + k·IQR]` (default `k = 1.5`)
- **`percentile`** — keeps rows between specified quantiles (e.g., 5th–95th)
- **`zscore`** — keeps rows within `mean ± threshold·std`

Categorical filtering reduces cardinality by keeping the top-N most frequent values.

**Why filter?**  
Extreme premium values and very rare distribution channels create poor generalisation.  
The filtered datasets vary per model (XGBoost, GLM, GAM, Linear) to allow independent tuning of cleaning thresholds.

In [311]:
def filter_middle_advanced(df, 
                          numeric_filters=None, 
                          categorical_filters=None,
                          keep_id_cols=None):
    """
    Advanced filtering with per-column configuration
    
    Parameters:
    -----------
    numeric_filters : dict
        {column_name: {'multiplier': 1.5, 'method': 'iqr'}}
    categorical_filters : dict
        {column_name: {'min_freq': 0.01, 'max_categories': 10, 'keep_values': [...]}}
    keep_id_cols : list
        Columns to preserve in output (like 'encoded_policy_number')
    """
    df_filtered = df.copy()
    
    if keep_id_cols is None:
        keep_id_cols = ['encoded_policy_number']
    
    # Numeric filtering
    if numeric_filters:
        for col, config in numeric_filters.items():
            multiplier = config.get('multiplier', 1.5)
            method = config.get('method', 'iqr')
            
            if method == 'iqr':
                Q1 = df[col].quantile(0.25)
                Q3 = df[col].quantile(0.75)
                IQR = Q3 - Q1
                lower = Q1 - multiplier * IQR
                upper = Q3 + multiplier * IQR
            elif method == 'percentile':
                lower = df[col].quantile(config.get('lower_q', 0.05))
                upper = df[col].quantile(config.get('upper_q', 0.95))
            elif method == 'zscore':
                mean = df[col].mean()
                std = df[col].std()
                threshold = config.get('threshold', 3)
                lower = mean - threshold * std
                upper = mean + threshold * std
            
            df_filtered = df_filtered[(df_filtered[col] >= lower) & (df_filtered[col] <= upper)]
            print(f"{col}: [{lower:.2f}, {upper:.2f}]")
    
    # Categorical filtering
    if categorical_filters:
        for col, config in categorical_filters.items():
            # Option 1: Explicit list of values to keep
            if 'keep_values' in config:
                valid_values = config['keep_values']
            # Option 2: By frequency or top N
            else:
                value_counts = df[col].value_counts()
                
                if 'max_categories' in config:
                    valid_values = value_counts.nlargest(config['max_categories']).index.tolist()
                elif 'min_freq' in config:
                    min_count = int(len(df) * config['min_freq'])
                    valid_values = value_counts[value_counts >= min_count].index.tolist()
                else:
                    valid_values = value_counts.index.tolist()
            
            df_filtered = df_filtered[df_filtered[col].isin(valid_values)]
            print(f"{col}: Kept {len(valid_values)} categories")
    
    print(f"\nTotal: {len(df)} → {len(df_filtered)} rows")
    
    return df_filtered


# Usage
df_middle = filter_middle_advanced(
    df_sub,
    numeric_filters={
        'X_age': {'multiplier': 1.5, 'method': 'iqr'},
        'X_policy_premium': {'multiplier': 2.0, 'method': 'iqr'},
        'X_upcoming_premium':{'multiplier':2.0,'method':'iqr'},
        'X_vehicle_age': {'method': 'percentile', 'lower_q': 0.05, 'upper_q': 0.95},
        'X_cur_renewal_perc':{'method': 'percentile', 'lower_q': 0.05, 'upper_q': 0.95}
    },
    categorical_filters={
        'X_distr_channel': {'max_categories':12},
        'X_risk_code': {'max_categories': 6},
        'X_vehicle_type': {'max_categories': 8}  # explicit list
    },
    keep_id_cols=['encoded_policy_number']
)

X_age: [8.50, 100.50]
X_policy_premium: [-14.06, 444.14]
X_upcoming_premium: [-15.07, 480.13]
X_vehicle_age: [4.00, 31.00]
X_cur_renewal_perc: [-0.02, 0.11]
X_distr_channel: Kept 12 categories
X_risk_code: Kept 6 categories
X_vehicle_type: Kept 8 categories

Total: 463400 → 194373 rows


## Section 5 — Model Architecture Definitions

Three model classes are defined here and reused across datasets:

### `ModelRegressor`
XGBoost regressor with **Optuna hyperparameter tuning**.  
- Objective: `reg:absoluteerror` (MAE-optimised)
- Tuned parameters: `learning_rate`, `max_depth`, `subsample`, `colsample_bytree`, `min_child_weight`
- Used to predict `X_policy_premium` (target `Y`)

### `ModelClassifier`
XGBoost binary classifier with **Optuna hyperparameter tuning**.  
- Metric: ROC-AUC
- Used to predict churn (`is_churn`), which is then converted to acceptance probability: `Z = 1 − is_churn`, `prob_acceptance = 1 − churn_proba`

### `ModelRegressorCV`  
Extended version of `ModelRegressor` that runs **fully nested Optuna + KFold CV** in a single call. Used for the policy premium regression CV experiment.

In [312]:

from sklearn import metrics
import pandas as pd
import numpy as np
import xgboost as xgb
import optuna
from optuna.trial import Trial
N_ESTIMATORS = 200
OBJECTIVE = 'reg:absoluteerror'
class ModelRegressor():
    
    def objective(self,trial : Trial, X_train : pd.DataFrame, y_train : pd.Series,X_val : pd.DataFrame, y_val : pd.Series)-> float:
        params = {
            "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.1, log=True),
            "max_depth": trial.suggest_int("max_depth", 1, 10),
            "subsample": trial.suggest_float("subsample", 0.05, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.05, 1.0),
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
        }
        model = xgb.XGBRegressor(**params,random_state=123,n_estimators=N_ESTIMATORS,early_stopping_rounds=5,objective = OBJECTIVE)
        model.fit(X_train, y_train,verbose=0,eval_set=[(X_val, y_val)])
        y_pred = model.predict(X_val)
        loss = metrics.mean_absolute_error(y_true=y_val,y_pred=y_pred)
        return loss
    
    def tuning(self,X_train:pd.DataFrame,y_train:pd.Series,X_val:pd.DataFrame, y_val:pd.Series,n_trials:int)->dict:
        y_train = y_train
        y_val = y_val
        study = optuna.create_study(study_name='Xgboost', direction='minimize')
        study.optimize(lambda trial: self.objective(trial, X_train, y_train, X_val, y_val), n_trials=n_trials)
        best_params = study.best_params
        return best_params
    
    def train(self,X_train:pd.DataFrame,y_train:pd.Series,X_val:pd.DataFrame, y_val:pd.Series,best_params:dict)->xgb.XGBRegressor:
        y_train = y_train
        y_val = y_val
        model = xgb.XGBRegressor(**best_params,random_state=123,n_estimators=N_ESTIMATORS,early_stopping_rounds=5,objective = OBJECTIVE)
        model.fit(X_train, y_train,verbose=0,eval_set=[(X_val, y_val)])
        return model

In [313]:
from sklearn import metrics
import pandas as pd
import xgboost as xgb
import optuna
from optuna.trial import Trial

N_ESTIMATORS = 200

class ModelClassifier():
    
    def objective(self,trial : Trial, X_train : pd.DataFrame, y_train : pd.Series,X_val : pd.DataFrame, y_val : pd.Series)-> float:
        params = {
            "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.1, log=True),
            "max_depth": trial.suggest_int("max_depth", 1, 10),
            "subsample": trial.suggest_float("subsample", 0.05, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.05, 1.0),
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
        }

        model = xgb.XGBClassifier(**params,random_state=123,n_estimators=N_ESTIMATORS,early_stopping_rounds=5)
        model.fit(X_train, y_train,verbose=0,eval_set=[(X_val, y_val)])
        y_score = model.predict_proba(X_val)[:,1]
        roc_auc = metrics.roc_auc_score(y_true=y_val,y_score=y_score)
        return roc_auc
    
    def tuning(self,X_train:pd.DataFrame,y_train:pd.Series,X_val:pd.DataFrame, y_val:pd.Series,n_trials:int)->dict:
        study = optuna.create_study(study_name='Xgboost', direction='maximize')
        study.optimize(lambda trial: self.objective(trial, X_train, y_train, X_val, y_val), n_trials=n_trials)
        best_params = study.best_params
        return best_params
    
    def train(self,X_train:pd.DataFrame,y_train:pd.Series,X_val:pd.DataFrame, y_val:pd.Series,best_params:dict)->xgb.XGBClassifier:
        model = xgb.XGBClassifier(**best_params,random_state= 123,n_estimators= N_ESTIMATORS,
                                  early_stopping_rounds= 5)
        model.fit(X_train, y_train,verbose=0,eval_set=[(X_val, y_val)])
        return model


In [314]:
# generate dset for class xgb and xgb reg

In [316]:
df_middle.columns

Index(['encoded_policy_number', 'X_age', 'X_bonus_malus_rating',
       'X_distr_channel', 'X_vehicle_type', 'X_ttm_claims', 'X_policy_count',
       'X_risk_code', 'X_vehicle_age', 'X_policy_tenure', 'X_policy_premium',
       'X_prev_renewal_perc', 'X_upcoming_premium', 'X_cur_renewal_perc',
       'year', 'is_churn'],
      dtype='object')

In [317]:
df_middle_class = filter_middle_advanced(
    df_sub,
    numeric_filters={
        'X_age': {'multiplier': 1.5, 'method': 'iqr'},
        'X_policy_premium': {'multiplier': 2.0, 'method': 'iqr'},
        'X_upcoming_premium':{'multiplier':2.0,'method':'iqr'},
        'X_vehicle_age': {'method': 'percentile', 'lower_q': 0.05, 'upper_q': 0.95},
        'X_cur_renewal_perc':{'method': 'percentile', 'lower_q': 0.05, 'upper_q': 0.95}
    },
    categorical_filters={
        'X_distr_channel': {'max_categories':12},
        'X_risk_code': {'max_categories': 6},
        'X_vehicle_type': {'max_categories': 8}  # explicit list
    },
    keep_id_cols=['encode_policy_number']
)

X_age: [8.50, 100.50]
X_policy_premium: [-14.06, 444.14]
X_upcoming_premium: [-15.07, 480.13]
X_vehicle_age: [4.00, 31.00]
X_cur_renewal_perc: [-0.02, 0.11]
X_distr_channel: Kept 12 categories
X_risk_code: Kept 6 categories
X_vehicle_type: Kept 8 categories

Total: 463400 → 194373 rows


In [318]:
def filter_middle_iqr(df, columns=None, multiplier=1.5):
    """Keep observations within Q1 - multiplier*IQR to Q3 + multiplier*IQR"""
    df_filtered = df.copy()
    
    # Use numeric columns only if not specified
    if columns is None:
        columns = df.select_dtypes(include=['number']).columns
        columns = [col for col in columns if col not in ['encoded_policy_number', 'is_churn']]
    
    for col in columns:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - multiplier * IQR
        upper = Q3 + multiplier * IQR
        df_filtered = df_filtered[(df_filtered[col] >= lower) & (df_filtered[col] <= upper)]
    
    return df_filtered



In [320]:
df_middle_class.columns

Index(['encoded_policy_number', 'X_age', 'X_bonus_malus_rating',
       'X_distr_channel', 'X_vehicle_type', 'X_ttm_claims', 'X_policy_count',
       'X_risk_code', 'X_vehicle_age', 'X_policy_tenure', 'X_policy_premium',
       'X_prev_renewal_perc', 'X_upcoming_premium', 'X_cur_renewal_perc',
       'year', 'is_churn'],
      dtype='object')

In [321]:
from sklearn import metrics
from sklearn.model_selection import KFold, cross_val_score
import pandas as pd
import numpy as np
import xgboost as xgb
import optuna
from optuna.trial import Trial

N_ESTIMATORS = 200
OBJECTIVE = 'reg:absoluteerror'
N_FOLDS = 5
RANDOM_STATE = 123

class ModelRegressorCV():
    
    def __init__(self, n_folds=N_FOLDS, random_state=RANDOM_STATE):
        self.n_folds = n_folds
        self.random_state = random_state
        self.best_params = None
        self.cv_scores = None
        self.final_model = None
        
    def objective(self, trial: Trial, X: pd.DataFrame, y: pd.Series) -> float:
        """Optuna objective with cross-validation"""
        params = {
            "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.1, log=True),
            "max_depth": trial.suggest_int("max_depth", 1, 10),
            "subsample": trial.suggest_float("subsample", 0.05, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.05, 1.0),
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
        }
        
        # Cross-validation with the suggested parameters
        kfold = KFold(n_splits=self.n_folds, shuffle=True, random_state=self.random_state)
        cv_scores = []
        
        for train_idx, val_idx in kfold.split(X):
            X_train_fold, X_val_fold = X.iloc[train_idx], X.iloc[val_idx]
            y_train_fold, y_val_fold = y.iloc[train_idx], y.iloc[val_idx]
            
            # Apply log transformation
            y_train_transformed = np.log(y_train_fold) + 1
            y_val_transformed = np.log(y_val_fold) + 1
            
            model = xgb.XGBRegressor(
                **params,
                random_state=self.random_state,
                n_estimators=N_ESTIMATORS,
                early_stopping_rounds=5,
                objective=OBJECTIVE
            )
            
            model.fit(
                X_train_fold, 
                y_train_transformed,
                verbose=0,
                eval_set=[(X_val_fold, y_val_transformed)]
            )
            
            y_pred = model.predict(X_val_fold)
            fold_mae = metrics.mean_absolute_error(y_val_transformed, y_pred)
            cv_scores.append(fold_mae)
        
        # Return mean CV score
        return np.mean(cv_scores)
    
    def tuning(self, X: pd.DataFrame, y: pd.Series, n_trials: int) -> dict:
        """Hyperparameter tuning with cross-validation"""
        print(f"Starting hyperparameter tuning with {n_trials} trials and {self.n_folds}-fold CV...")
        
        study = optuna.create_study(study_name='XGBoost_CV', direction='minimize')
        study.optimize(
            lambda trial: self.objective(trial, X, y), 
            n_trials=n_trials,
            show_progress_bar=True
        )
        
        self.best_params = study.best_params
        print(f"\nBest CV MAE: {study.best_value:.6f}")
        print(f"Best parameters: {self.best_params}")
        
        return self.best_params
    
    def train_with_cv(self, X: pd.DataFrame, y: pd.Series, params: dict = None) -> tuple:
        """Train model with cross-validation and return CV scores"""
        if params is None:
            if self.best_params is None:
                raise ValueError("No parameters provided. Run tuning() first or provide params.")
            params = self.best_params
        
        print(f"\nTraining with {self.n_folds}-fold cross-validation...")
        kfold = KFold(n_splits=self.n_folds, shuffle=True, random_state=self.random_state)
        
        cv_scores = []
        fold_models = []
        
        for fold_num, (train_idx, val_idx) in enumerate(kfold.split(X), 1):
            X_train_fold, X_val_fold = X.iloc[train_idx], X.iloc[val_idx]
            y_train_fold, y_val_fold = y.iloc[train_idx], y.iloc[val_idx]
            
            # Apply log transformation
            y_train_transformed = y_train_fold 
            y_val_transformed = y_val_fold 
            
            model = xgb.XGBRegressor(
                **params,
                random_state=self.random_state,
                n_estimators=N_ESTIMATORS,
                early_stopping_rounds=5,
                objective=OBJECTIVE
            )
            
            model.fit(
                X_train_fold, 
                y_train_transformed,
                verbose=0,
                eval_set=[(X_val_fold, y_val_transformed)]
            )
            
            y_pred = model.predict(X_val_fold)
            fold_mae = metrics.mean_absolute_error(y_val_transformed, y_pred)
            cv_scores.append(fold_mae)
            fold_models.append(model)
            
            print(f"Fold {fold_num}: MAE = {fold_mae:.6f}")
        
        self.cv_scores = cv_scores
        print(f"\nMean CV MAE: {np.mean(cv_scores):.6f} (+/- {np.std(cv_scores):.6f})")
        
        return fold_models, cv_scores
    
    def train_final(self, X: pd.DataFrame, y: pd.Series, params: dict = None) -> xgb.XGBRegressor:
        """Train final model on full dataset"""
        if params is None:
            if self.best_params is None:
                raise ValueError("No parameters provided. Run tuning() first or provide params.")
            params = self.best_params
        
        print("\nTraining final model on full dataset...")
        y_transformed = np.log(y) + 1
        
        # Use a portion for validation (20%)
        split_idx = int(len(X) * 0.8)
        X_train, X_val = X.iloc[:split_idx], X.iloc[split_idx:]
        y_train, y_val = y_transformed.iloc[:split_idx], y_transformed.iloc[split_idx:]
        
        model = xgb.XGBRegressor(
            **params,
            random_state=self.random_state,
            n_estimators=N_ESTIMATORS,
            early_stopping_rounds=5,
            objective=OBJECTIVE
        )
        
        model.fit(
            X_train, 
            y_train,
            verbose=0,
            eval_set=[(X_val, y_val)]
        )
        
        self.final_model = model
        print("Final model training complete!")
        
        return model
    
    def predict(self, X: pd.DataFrame, inverse_transform: bool = True) -> np.ndarray:
        """Make predictions using the final model"""
        if self.final_model is None:
            raise ValueError("No model trained. Run train_final() first.")
        
        predictions = self.final_model.predict(X)
        
        if inverse_transform:
            # Inverse log transformation
            predictions = predictions
        
        return predictions


In [322]:
from sklearn.model_selection import KFold, cross_val_score
import pandas as pd
import numpy as np
import xgboost as xgb
import optuna
from optuna.trial import Trial

N_ESTIMATORS = 200
OBJECTIVE = 'reg:absoluteerror'
N_FOLDS = 5
RANDOM_STATE = 123

class ModelRegressorCV():
    
    def __init__(self, n_folds=N_FOLDS, random_state=RANDOM_STATE):
        self.n_folds = n_folds
        self.random_state = random_state
        self.best_params = None
        self.cv_scores = None
        self.final_model = None
        
    def objective(self, trial: Trial, X: pd.DataFrame, y: pd.Series) -> float:
        """Optuna objective with cross-validation"""
        params = {
            "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.1, log=True),
            "max_depth": trial.suggest_int("max_depth", 1, 10),
            "subsample": trial.suggest_float("subsample", 0.05, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.05, 1.0),
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
        }
        
        # Cross-validation with the suggested parameters
        kfold = KFold(n_splits=self.n_folds, shuffle=True, random_state=self.random_state)
        cv_scores = []
        
        for train_idx, val_idx in kfold.split(X):
            X_train_fold, X_val_fold = X.iloc[train_idx], X.iloc[val_idx]
            y_train_fold, y_val_fold = y.iloc[train_idx], y.iloc[val_idx]
            
            # Apply log transformation
            y_train_transformed = np.log(y_train_fold) + 1
            y_val_transformed = np.log(y_val_fold) + 1
            
            model = xgb.XGBRegressor(
                **params,
                random_state=self.random_state,
                n_estimators=N_ESTIMATORS,
                early_stopping_rounds=5,
                objective=OBJECTIVE
            )
            
            model.fit(
                X_train_fold, 
                y_train_transformed,
                verbose=0,
                eval_set=[(X_val_fold, y_val_transformed)]
            )
            
            y_pred = model.predict(X_val_fold)
            fold_mae = metrics.mean_absolute_error(y_val_transformed, y_pred)
            cv_scores.append(fold_mae)
        
        # Return mean CV score
        return np.mean(cv_scores)
    
    def tuning(self, X: pd.DataFrame, y: pd.Series, n_trials: int) -> dict:
        """Hyperparameter tuning with cross-validation"""
        print(f"Starting hyperparameter tuning with {n_trials} trials and {self.n_folds}-fold CV...")
        
        study = optuna.create_study(study_name='XGBoost_CV', direction='minimize')
        study.optimize(
            lambda trial: self.objective(trial, X, y), 
            n_trials=n_trials,
            show_progress_bar=True
        )
        
        self.best_params = study.best_params
        print(f"\nBest CV MAE: {study.best_value:.6f}")
        print(f"Best parameters: {self.best_params}")
        
        return self.best_params
    
    def train_with_cv(self, X: pd.DataFrame, y: pd.Series, params: dict = None) -> tuple:
        """Train model with cross-validation and return CV scores"""
        if params is None:
            if self.best_params is None:
                raise ValueError("No parameters provided. Run tuning() first or provide params.")
            params = self.best_params
        
        print(f"\nTraining with {self.n_folds}-fold cross-validation...")
        kfold = KFold(n_splits=self.n_folds, shuffle=True, random_state=self.random_state)
        
        cv_scores = []
        fold_models = []
        
        for fold_num, (train_idx, val_idx) in enumerate(kfold.split(X), 1):
            X_train_fold, X_val_fold = X.iloc[train_idx], X.iloc[val_idx]
            y_train_fold, y_val_fold = y.iloc[train_idx], y.iloc[val_idx]
            
            # Apply log transformation
            y_train_transformed = np.log(y_train_fold) + 1
            y_val_transformed = np.log(y_val_fold) + 1
            
            model = xgb.XGBRegressor(
                **params,
                random_state=self.random_state,
                n_estimators=N_ESTIMATORS,
                early_stopping_rounds=5,
                objective=OBJECTIVE
            )
            
            model.fit(
                X_train_fold, 
                y_train_transformed,
                verbose=0,
                eval_set=[(X_val_fold, y_val_transformed)]
            )
            
            y_pred = model.predict(X_val_fold)
            fold_mae = metrics.mean_absolute_error(y_val_transformed, y_pred)
            cv_scores.append(fold_mae)
            fold_models.append(model)
            
            print(f"Fold {fold_num}: MAE = {fold_mae:.6f}")
        
        self.cv_scores = cv_scores
        print(f"\nMean CV MAE: {np.mean(cv_scores):.6f} (+/- {np.std(cv_scores):.6f})")
        
        return fold_models, cv_scores
    
    def train_final(self, X: pd.DataFrame, y: pd.Series, params: dict = None) -> xgb.XGBRegressor:
        """Train final model on full dataset"""
        if params is None:
            if self.best_params is None:
                raise ValueError("No parameters provided. Run tuning() first or provide params.")
            params = self.best_params
        
        print("\nTraining final model on full dataset...")
        y_transformed = np.log(y) + 1
        
        # Use a portion for validation (20%)
        split_idx = int(len(X) * 0.8)
        X_train, X_val = X.iloc[:split_idx], X.iloc[split_idx:]
        y_train, y_val = y_transformed.iloc[:split_idx], y_transformed.iloc[split_idx:]
        
        model = xgb.XGBRegressor(
            **params,
            random_state=self.random_state,
            n_estimators=N_ESTIMATORS,
            early_stopping_rounds=5,
            objective=OBJECTIVE
        )
        
        model.fit(
            X_train, 
            y_train,
            verbose=0,
            eval_set=[(X_val, y_val)]
        )
        
        self.final_model = model
        print("Final model training complete!")
        
        return model
    
    def predict(self, X: pd.DataFrame, inverse_transform: bool = True) -> np.ndarray:
        """Make predictions using the final model"""
        if self.final_model is None:
            raise ValueError("No model trained. Run train_final() first.")
        
        predictions = self.final_model.predict(X)
        
        if inverse_transform:
            # Inverse log transformation
            predictions = np.exp(predictions - 1)
        
        return predictions

In [323]:
# encode category feats

In [324]:
cat_cols = list(df_middle.select_dtypes('object').columns)

from sklearn.preprocessing import LabelEncoder
for cat in cat_cols:
    le = LabelEncoder()
    le.fit(df_middle[cat])
    df_middle[cat] = le.transform(df_middle[cat])
    df_middle_class[cat] = le.transform(df_middle_class[cat])


## Section 6 — Dataset Preparation for XGBoost

Two filtered datasets are created from `df_sub`:

- **`df_middle`** — used for the **premium regressor** (target: `X_policy_premium`)  
- **`df_middle_class`** — used for the **churn classifier** (target: `is_churn`)

Both use identical `filter_middle_advanced` thresholds.  
Categorical columns (`X_distr_channel`, `X_risk_code`, `X_vehicle_type`) are **label-encoded** here so XGBoost can consume them directly.

The premium change ratio `U` is computed as:  
$$U = \frac{\text{upcoming\_premium} - \text{policy\_premium}}{\text{policy\_premium}}$$  
This is a core feature driving churn behaviour: larger premium increases lead to higher churn probability.  
`U` is later adjusted to `1 + U` (i.e., the *uplift factor*) for the optimisation module.

In [325]:
df_middle.columns

Index(['encoded_policy_number', 'X_age', 'X_bonus_malus_rating',
       'X_distr_channel', 'X_vehicle_type', 'X_ttm_claims', 'X_policy_count',
       'X_risk_code', 'X_vehicle_age', 'X_policy_tenure', 'X_policy_premium',
       'X_prev_renewal_perc', 'X_upcoming_premium', 'X_cur_renewal_perc',
       'year', 'is_churn'],
      dtype='object')

In [326]:
df_middle['U'] = (df_middle['X_upcoming_premium'] - df_middle['X_policy_premium']) / df_middle['X_policy_premium']
df_middle_class['U'] = (df_middle_class['X_upcoming_premium'] - df_middle_class['X_policy_premium']) / df_middle_class['X_policy_premium']

In [327]:
df_middle_class['U'].describe()

count    194373.000000
mean          0.083095
std           0.029994
min          -0.001829
25%           0.061570
50%           0.094024
75%           0.103913
max           0.418037
Name: U, dtype: float64

In [328]:
df_middle.columns

Index(['encoded_policy_number', 'X_age', 'X_bonus_malus_rating',
       'X_distr_channel', 'X_vehicle_type', 'X_ttm_claims', 'X_policy_count',
       'X_risk_code', 'X_vehicle_age', 'X_policy_tenure', 'X_policy_premium',
       'X_prev_renewal_perc', 'X_upcoming_premium', 'X_cur_renewal_perc',
       'year', 'is_churn', 'U'],
      dtype='object')

In [329]:
model_features_class = ['X_age','X_bonus_malus_rating','X_distr_channel','X_vehicle_type','X_ttm_claims','X_policy_count','X_risk_code','X_vehicle_age','X_policy_tenure','X_policy_premium','U']

model_features_reg = ['X_age','X_bonus_malus_rating','X_distr_channel','X_vehicle_type','X_ttm_claims','X_policy_count','X_risk_code','X_vehicle_age','X_policy_tenure']

In [330]:
from sklearn.model_selection import KFold, cross_val_score
from sklearn import metrics
import pandas as pd
import numpy as np
import xgboost as xgb

# Assuming 'df' is your dataframe with the columns you mentioned
# First, filter to middle of distribution (using IQR method)
def filter_middle_distribution(df, multiplier=1.5):
    """Remove outliers using IQR method"""
    df_filtered = df.copy()
    
    # Select numeric columns, exclude ID and target
    numeric_cols = df.select_dtypes(include=['number']).columns
    numeric_cols = [col for col in numeric_cols if col not in ['encoded_policy_number', 'is_churn']]
    
    for col in numeric_cols:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - multiplier * IQR
        upper = Q3 + multiplier * IQR
        df_filtered = df_filtered[(df_filtered[col] >= lower) & (df_filtered[col] <= upper)]
    
    print(f"Original rows: {len(df)}, Filtered rows: {len(df_filtered)}")
    return df_filtered

# Apply filtering


# Separate features and target
X = df_middle.loc[:, model_features_reg]
y = df_middle['X_upcoming_premium']

# Initialize the model
model_regressor = ModelRegressor()

# Cross-validation approach 1: Using sklearn's cross_val_score
kfold = KFold(n_splits=5, shuffle=True, random_state=123)

# Simple CV without hyperparameter tuning
default_params = {
    'learning_rate': 0.01,
    'max_depth': 5,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'min_child_weight': 1,
    'random_state': 123,
    'n_estimators': 200,
    'objective': 'reg:absoluteerror'
}

xgb_model = xgb.XGBRegressor(**default_params, enable_categorical=True)

# Perform cross-validation
cv_scores = cross_val_score(
    xgb_model, 
    X, 
    y,  # Apply log transformation
    cv=kfold, 
    scoring='neg_mean_absolute_error',
    n_jobs=-1
)

print(f"CV MAE Scores: {-cv_scores}")
print(f"Mean CV MAE: {-cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

CV MAE Scores: [35.9154529  35.24502254 35.43045504 35.25128434 35.34746308]
Mean CV MAE: 35.4379 (+/- 0.2483)


In [331]:
df_middle.columns

Index(['encoded_policy_number', 'X_age', 'X_bonus_malus_rating',
       'X_distr_channel', 'X_vehicle_type', 'X_ttm_claims', 'X_policy_count',
       'X_risk_code', 'X_vehicle_age', 'X_policy_tenure', 'X_policy_premium',
       'X_prev_renewal_perc', 'X_upcoming_premium', 'X_cur_renewal_perc',
       'year', 'is_churn', 'U'],
      dtype='object')

## Section 7 — XGBoost Policy Premium Regression (5-Fold OOF)

**Target variable:** `X_policy_premium` (current active premium, `Y`)  
**Features:** `model_features_reg` — 9 policy/client covariates (excluding premium-related features)

`cross_validate_with_oof_predictions` runs a **nested cross-validation**:
1. Outer KFold (5 splits) — for generating honest out-of-fold (OOF) predictions  
2. Inner 80/20 split — for Optuna hyperparameter tuning within each fold  

**Outputs stored in `df_middle`:**
- `fold_number` — which fold each row was validated in
- `oof_prediction` (`Y_hat`) — predicted policy premium  
- `oof_residual` / `oof_absolute_error` — residual diagnostics  

These OOF predictions are used as the regressor output in `df_exp_financial_loss_xgb_black_box.csv`.

In [333]:
from sklearn.model_selection import KFold
import pandas as pd
import numpy as np

def cross_validate_with_oof_predictions(X, y, n_splits=5, n_trials=30):
    """
    Perform cross-validation and generate out-of-fold predictions
    Returns: DataFrame with original data + fold number + OOF predictions
    """
    kfold = KFold(n_splits=n_splits, shuffle=True, random_state=123)
    
    # Initialize arrays to store fold numbers and OOF predictions
    fold_numbers = np.zeros(len(X))
    oof_predictions = np.zeros(len(X))
    
    fold_results = []
    best_params_per_fold = []
    
    for fold_idx, (train_idx, val_idx) in enumerate(kfold.split(X)):
        print(f"\n{'='*50}")
        print(f"Fold {fold_idx + 1}/{n_splits}")
        print(f"{'='*50}")
        
        # Assign fold number to validation indices
        fold_numbers[val_idx] = fold_idx + 1
        
        # Split data for this fold
        X_train_fold, X_val_fold = X.iloc[train_idx], X.iloc[val_idx]
        y_train_fold, y_val_fold = y.iloc[train_idx], y.iloc[val_idx]
        
        # Further split training data for Optuna tuning
        from sklearn.model_selection import train_test_split
        X_train_tune, X_val_tune, y_train_tune, y_val_tune = train_test_split(
            X_train_fold, y_train_fold, test_size=0.2, random_state=123
        )
        
        # Tune hyperparameters
        model_reg = ModelRegressor()
        
        
        best_params = model_reg.tuning(
            X_train_tune, y_train_tune, 
            X_val_tune, y_val_tune, 
            n_trials=n_trials
        )
        
        
        print(f"Best params for fold {fold_idx + 1}: {best_params}")
        best_params_per_fold.append(best_params)
        
        # Train final model on full training fold
        final_model = model_reg.train(
            X_train_fold, y_train_fold,
            X_val_fold, y_val_fold,
            best_params
        )
        
        # Generate predictions for validation fold (log scale)
        y_pred_log = final_model.predict(X_val_fold)
        
        # Store OOF predictions (in original scale)
        oof_predictions[val_idx] = y_pred_log
        
        # Calculate metrics
        y_val_transformed = y_val_fold
        mae_log = metrics.mean_absolute_error(y_val_transformed, y_pred_log)
        
        # Original scale metrics
        mae_original = metrics.mean_absolute_error(y_val_fold, oof_predictions[val_idx])
        
        fold_results.append({
            'fold': fold_idx + 1,
            'mae_log_scale': mae_log,
            'mae_original_scale': mae_original,
            'n_train': len(train_idx),
            'n_val': len(val_idx)
        })
        
        print(f"Fold {fold_idx + 1} MAE (log scale): {mae_log:.4f}")
        print(f"Fold {fold_idx + 1} MAE (original scale): {mae_original:.4f}")
    
    # Create result dataframe
    results_df = pd.DataFrame(fold_results)
    
    # Summary
    print(f"\n{'='*50}")
    print("Cross-Validation Summary")
    print(f"{'='*50}")
    print(f"Mean MAE (log scale): {results_df['mae_log_scale'].mean():.4f} (+/- {results_df['mae_log_scale'].std():.4f})")
    print(f"Mean MAE (original scale): {results_df['mae_original_scale'].mean():.4f} (+/- {results_df['mae_original_scale'].std():.4f})")
    
    # Calculate overall OOF score
    overall_oof_mae = metrics.mean_absolute_error(y, oof_predictions)
    print(f"\nOverall OOF MAE: {overall_oof_mae:.4f}")
    
    return fold_numbers, oof_predictions, results_df, best_params_per_fold


# Usage example:
# Assuming df_filtered is your filtered dataframe
X = df_middle.loc[:, model_features_reg]
y = df_middle['X_policy_premium']

# Run cross-validation and get OOF predictions
fold_numbers, oof_predictions, results_df, best_params_list = cross_validate_with_oof_predictions(
    X, y, n_splits=5, n_trials=30
)

# Add columns to the original filtered dataframe
df_middle['fold_number'] = fold_numbers.astype(int)
df_middle['oof_prediction'] = oof_predictions

# Calculate residuals
df_middle['oof_residual'] = df_middle['X_policy_premium'] - df_middle['oof_prediction']
df_middle['oof_absolute_error'] = np.abs(df_middle['oof_residual'])



# Display sample
print("\nSample of dataframe with OOF predictions:")
print(df_middle[['encoded_policy_number', 'is_churn', 'fold_number', 
                    'oof_prediction', 'oof_residual']].head(10))

# Summary statistics by fold
print("\nOOF Performance by Fold:")
fold_summary = df_middle.groupby('fold_number').agg({
    'is_churn': 'count',
    'oof_absolute_error': 'mean'
}).rename(columns={'is_churn': 'count', 'oof_absolute_error': 'mae'})
print(fold_summary)

[I 2026-03-31 09:34:10,565] A new study created in memory with name: Xgboost



Fold 1/5


[I 2026-03-31 09:34:15,242] Trial 0 finished with value: 35.31442955440254 and parameters: {'learning_rate': 0.008196409789504, 'max_depth': 8, 'subsample': 0.44613004381289956, 'colsample_bytree': 0.43581056141495667, 'min_child_weight': 4}. Best is trial 0 with value: 35.31442955440254.
[I 2026-03-31 09:34:21,231] Trial 1 finished with value: 40.925125837107174 and parameters: {'learning_rate': 0.001856373194872954, 'max_depth': 1, 'subsample': 0.6812599677930663, 'colsample_bytree': 0.2784029066031313, 'min_child_weight': 8}. Best is trial 0 with value: 35.31442955440254.
[I 2026-03-31 09:34:24,814] Trial 2 finished with value: 37.439017607291554 and parameters: {'learning_rate': 0.007017761474311904, 'max_depth': 6, 'subsample': 0.3687838856545533, 'colsample_bytree': 0.31045880128781733, 'min_child_weight': 3}. Best is trial 0 with value: 35.31442955440254.
[I 2026-03-31 09:34:29,011] Trial 3 finished with value: 39.61213976453853 and parameters: {'learning_rate': 0.00129176390431

Best params for fold 1: {'learning_rate': 0.08183929751834249, 'max_depth': 6, 'subsample': 0.9011046527832598, 'colsample_bytree': 0.7194125919171591, 'min_child_weight': 17}


[I 2026-03-31 09:36:38,506] A new study created in memory with name: Xgboost


...

Best params for fold 4: {'learning_rate': 0.09941117810726005, 'max_depth': 5, 'subsample': 0.781827600058718, 'colsample_bytree': 0.6397640277685516, 'min_child_weight': 17}


[I 2026-03-31 09:43:37,933] A new study created in memory with name: Xgboost


Fold 4 MAE (log scale): 30.3800
Fold 4 MAE (original scale): 30.3800

Fold 5/5


[I 2026-03-31 09:43:41,808] Trial 0 finished with value: 34.701354916027405 and parameters: {'learning_rate': 0.005102080518360063, 'max_depth': 7, 'subsample': 0.54095115355461, 'colsample_bytree': 0.8180156810298919, 'min_child_weight': 3}. Best is trial 0 with value: 34.701354916027405.
[I 2026-03-31 09:43:45,242] Trial 1 finished with value: 40.80846708497035 and parameters: {'learning_rate': 0.0013323819977196544, 'max_depth': 8, 'subsample': 0.1817351567996618, 'colsample_bytree': 0.4244761444915307, 'min_child_weight': 10}. Best is trial 0 with value: 34.701354916027405.
[I 2026-03-31 09:43:49,010] Trial 2 finished with value: 36.50238662494583 and parameters: {'learning_rate': 0.016719803142563595, 'max_depth': 1, 'subsample': 0.37622658181559104, 'colsample_bytree': 0.45596330148623526, 'min_child_weight': 5}. Best is trial 0 with value: 34.701354916027405.
[I 2026-03-31 09:43:51,995] Trial 3 finished with value: 34.196836291721354 and parameters: {'learning_rate': 0.017815099

Best params for fold 5: {'learning_rate': 0.060545790668976, 'max_depth': 6, 'subsample': 0.9915182263914608, 'colsample_bytree': 0.6239478826695686, 'min_child_weight': 14}
Fold 5 MAE (log scale): 30.4861
Fold 5 MAE (original scale): 30.4861

Cross-Validation Summary
Mean MAE (log scale): 30.5799 (+/- 0.2693)
Mean MAE (original scale): 30.5799 (+/- 0.2693)

Overall OOF MAE: 30.5799

Sample of dataframe with OOF predictions:
     encoded_policy_number  is_churn  fold_number  oof_prediction  \
7                        5     False            5      193.966034   
16                      85     False            4      162.414307   
24                      93     False            1      168.083298   
32                     119     False            5      175.712479   
118                    395     False            5      210.380798   
139                    470     False            3      210.291870   
146                    478     False            2      178.864288   
154                

## Section 8 — XGBoost Churn Classification (5-Fold OOF)

**Target variable:** `is_churn` (binary, 0 = retained, 1 = churned)  
**Features:** `model_features_class` — 11 features including `X_policy_premium` and `U`

`cross_validate_churn_with_oof_predictions` follows the same nested structure as the regression CV:
1. Outer 5-fold for OOF predictions  
2. Inner 80/20 split for Optuna tuning (maximise ROC-AUC)

**Outputs stored in `df_middle_class`:**
- `fold_number` — fold assignment  
- `oof_prediction` — OOF churn probability (output of `predict_proba[:, 1]`)
- `prob_acceptance = 1 − oof_prediction` — probability the client accepts (does not churn)
- `Z = 1 − is_churn` — binary acceptance indicator

The `prob_acceptance` column is the key output consumed by the pricing optimiser.

In [334]:
from sklearn.model_selection import KFold
import pandas as pd
import numpy as np

def cross_validate_churn_with_oof_predictions(X, y, n_splits=5, n_trials=30):
    """
    Perform cross-validation and generate out-of-fold predictions
    Returns: DataFrame with original data + fold number + OOF predictions
    """
    kfold = KFold(n_splits=n_splits, shuffle=True, random_state=123)
    
    # Initialize arrays to store fold numbers and OOF predictions
    fold_numbers = np.zeros(len(X))
    oof_predictions = np.zeros(len(X))
    
    fold_results = []
    best_params_per_fold = []
    
    for fold_idx, (train_idx, val_idx) in enumerate(kfold.split(X)):
        print(f"\n{'='*50}")
        print(f"Fold {fold_idx + 1}/{n_splits}")
        print(f"{'='*50}")
        
        # Assign fold number to validation indices
        fold_numbers[val_idx] = fold_idx + 1
        
        # Split data for this fold
        X_train_fold, X_val_fold = X.iloc[train_idx], X.iloc[val_idx]
        y_train_fold, y_val_fold = y.iloc[train_idx], y.iloc[val_idx]
        
        # Further split training data for Optuna tuning
        from sklearn.model_selection import train_test_split
        X_train_tune, X_val_tune, y_train_tune, y_val_tune = train_test_split(
            X_train_fold, y_train_fold, test_size=0.2, random_state=123
        )
        
        # Tune hyperparameters
        model_class = ModelClassifier()
        
        
        best_params = model_class.tuning(
            X_train_tune, y_train_tune, 
            X_val_tune, y_val_tune, 
            n_trials=n_trials
        )
        
        
        print(f"Best params for fold {fold_idx + 1}: {best_params}")
        best_params_per_fold.append(best_params)
        
        # Train final model on full training fold
        final_model = model_class.train(
            X_train_fold, y_train_fold,
            X_val_fold, y_val_fold,best_params)
        # Generate predictions for validation fold (log scale)
        y_pred = final_model.predict(X_val_fold)
        y_pred_proba = final_model.predict_proba(X_val_fold)[:,1]
        
        roc_auc = metrics.roc_auc_score(y_true=y_val_fold, y_score=y_pred_proba)
        # Store OOF predictions (in original scale)
        oof_predictions[val_idx] = y_pred_proba
        
        
        # Calculate metrics
        
        
        

        
        fold_results.append({
            'fold': fold_idx + 1,
            'roc_auc': roc_auc,
            'n_train': len(train_idx),
            'n_val': len(val_idx)
        })
        
        print(f"Fold {fold_idx + 1} AUC (log scale): {roc_auc:.4f}")
    
    # Create result dataframe
    results_df = pd.DataFrame(fold_results)
    
    # Summary
    print(f"\n{'='*50}")
    print("Cross-Validation Summary")


    
    return fold_numbers, oof_predictions, results_df, best_params_per_fold


# Usage example:
# Assuming df_filtered is your filtered dataframe
X = df_middle_class.loc[:,model_features_class]
y = df_middle_class['is_churn'].astype(int)

# Run cross-validation and get OOF predictions
fold_numbers, oof_predictions, results_df, best_params_list = cross_validate_churn_with_oof_predictions(
    X, y, n_splits=5, n_trials=30
)


df_middle_class['fold_number'] = fold_numbers.astype(int)
df_middle_class['oof_prediction'] = oof_predictions
# Add columns to the original filtered dataframe




[I 2026-03-31 09:46:04,443] A new study created in memory with name: Xgboost



Fold 1/5


[I 2026-03-31 09:46:05,900] Trial 0 finished with value: 0.6261414839152228 and parameters: {'learning_rate': 0.0012058415550974786, 'max_depth': 2, 'subsample': 0.37949398075941904, 'colsample_bytree': 0.9529677506880325, 'min_child_weight': 9}. Best is trial 0 with value: 0.6261414839152228.
[I 2026-03-31 09:46:07,057] Trial 1 finished with value: 0.6614852440284598 and parameters: {'learning_rate': 0.06263871684362121, 'max_depth': 5, 'subsample': 0.312680290738838, 'colsample_bytree': 0.3612661973111024, 'min_child_weight': 19}. Best is trial 1 with value: 0.6614852440284598.
[I 2026-03-31 09:46:08,156] Trial 2 finished with value: 0.6101319572112052 and parameters: {'learning_rate': 0.001109535213460959, 'max_depth': 1, 'subsample': 0.9944986867204899, 'colsample_bytree': 0.6734885896456438, 'min_child_weight': 20}. Best is trial 1 with value: 0.6614852440284598.
[I 2026-03-31 09:46:09,305] Trial 3 finished with value: 0.6233499766443613 and parameters: {'learning_rate': 0.0053529

Best params for fold 1: {'learning_rate': 0.09786135247299849, 'max_depth': 4, 'subsample': 0.5301003479615278, 'colsample_bytree': 0.565844903397824, 'min_child_weight': 3}


[I 2026-03-31 09:46:37,816] A new study created in memory with name: Xgboost


...

Best params for fold 4: {'learning_rate': 0.049391224441331975, 'max_depth': 5, 'subsample': 0.8704956989667747, 'colsample_bytree': 0.3722389124881974, 'min_child_weight': 5}


[I 2026-03-31 09:49:12,708] A new study created in memory with name: Xgboost


Fold 4 AUC (log scale): 0.6671

Fold 5/5


[I 2026-03-31 09:49:14,259] Trial 0 finished with value: 0.6587502010749874 and parameters: {'learning_rate': 0.004077972478448117, 'max_depth': 6, 'subsample': 0.08184093471646892, 'colsample_bytree': 0.755733271148202, 'min_child_weight': 10}. Best is trial 0 with value: 0.6587502010749874.
[I 2026-03-31 09:49:15,443] Trial 1 finished with value: 0.6639298666882857 and parameters: {'learning_rate': 0.03771973017069119, 'max_depth': 4, 'subsample': 0.9120177794120664, 'colsample_bytree': 0.2872658921640067, 'min_child_weight': 20}. Best is trial 1 with value: 0.6639298666882857.
[I 2026-03-31 09:49:17,247] Trial 2 finished with value: 0.6565005969701869 and parameters: {'learning_rate': 0.002345109566115772, 'max_depth': 5, 'subsample': 0.12587084968749856, 'colsample_bytree': 0.7709705506850856, 'min_child_weight': 10}. Best is trial 1 with value: 0.6639298666882857.
[I 2026-03-31 09:49:18,631] Trial 3 finished with value: 0.6494174949833494 and parameters: {'learning_rate': 0.003730

Best params for fold 5: {'learning_rate': 0.07161720971939052, 'max_depth': 3, 'subsample': 0.7596544211126036, 'colsample_bytree': 0.9549545566023544, 'min_child_weight': 8}
Fold 5 AUC (log scale): 0.6736

Cross-Validation Summary


## Section 9 — XGBoost Output: Acceptance Probability & Expected Financial Loss

Post-CV, the following export steps are performed:

### `df_acceptance_xgb_black_box.csv`
Selected columns from `df_middle_class`:  
`id, X_age, X_bonus_malus_rating, X_distr_channel, X_vehicle_type, X_ttm_claims, X_policy_count, X_risk_code, X_vehicle_age, X_policy_tenure, X_policy_premium, U, prob_acceptance, Z`

- `prob_acceptance` = acceptance probability predicted by the XGBoost classifier  
- `U` = premium uplift factor (`1 + relative_change`)  
- `Z` = binary acceptance outcome (ground truth)

### `df_exp_financial_loss_xgb_black_box.csv`
Selected columns from `df_middle`:  
`id, ..., Y (actual premium), U, Y_hat (predicted premium)`

- `Y_hat` = XGBoost regressor's OOF premium prediction  
- Used to compute expected financial loss = `Y_hat × (1 − prob_acceptance)`

In [335]:
df_middle_class['Z'] = 1- df_middle_class['is_churn'].astype(int)

In [336]:
model_features_class

['X_age',
 'X_bonus_malus_rating',
 'X_distr_channel',
 'X_vehicle_type',
 'X_ttm_claims',
 'X_policy_count',
 'X_risk_code',
 'X_vehicle_age',
 'X_policy_tenure',
 'X_policy_premium',
 'U']

In [337]:
['id', 'X_age','X_bonus_malus_rating','X_distr_channel','X_vehicle_type','X_ttm_claims','X_policy_count','X_risk_code','X_vehicle_age',
                           'X_policy_tenure','X_policy_premium', 'U','fold_number','churn_prediction','prob_acceptance','Z']

['id',
 'X_age',
 'X_bonus_malus_rating',
 'X_distr_channel',
 'X_vehicle_type',
 'X_ttm_claims',
 'X_policy_count',
 'X_risk_code',
 'X_vehicle_age',
 'X_policy_tenure',
 'X_policy_premium',
 'U',
 'fold_number',
 'churn_prediction',
 'prob_acceptance',
 'Z']

In [338]:
df_middle_class['prob_acceptance'] = 1 - df_middle_class['oof_prediction']

In [342]:
df_middle_class.columns

Index(['encoded_policy_number', 'X_age', 'X_bonus_malus_rating',
       'X_distr_channel', 'X_vehicle_type', 'X_ttm_claims', 'X_policy_count',
       'X_risk_code', 'X_vehicle_age', 'X_policy_tenure', 'X_policy_premium',
       'X_prev_renewal_perc', 'X_upcoming_premium', 'X_cur_renewal_perc',
       'year', 'is_churn', 'U', 'fold_number', 'oof_prediction', 'Z',
       'prob_acceptance'],
      dtype='object')

In [341]:
df_middle.columns = ['id', 'X_age','X_bonus_malus_rating','X_distr_channel','X_vehicle_type','X_ttm_claims','X_policy_count','X_risk_code','X_vehicle_age','X_policy_tenure',
                     'Y','X_prev_renewal_perc','X_upcoming_premium','X_cur_renewal_perc','X_year','1-Z','U','fold_number','Y_hat','Y_residual','Y_absolute_error']

In [343]:
df_middle_class.columns = ['id', 'X_age','X_bonus_malus_rating','X_distr_channel','X_vehicle_type','X_ttm_claims','X_policy_count','X_risk_code',"X_vehicle_age",
                           'X_policy_tenure','X_policy_premium','X_prev_renewal_perc','X_upcoming_premium','X_cur_renewal_perc','X_year','1-Z', 'U','fold_number','churn_prediction','Z','prob_acceptance']

In [344]:
df_middle_class['U'] = 1 + df_middle_class['U']

In [345]:
df_middle_class['U']

7          1.123099
16         1.107047
24         1.096895
32         1.097209
118        1.102011
             ...   
3370174    1.062686
3370323    1.087288
3370329    1.087051
3370407    1.087493
3370452    1.087271
Name: U, Length: 194373, dtype: float64

In [346]:
['id', 'X_age','X_bonus_malus_rating','X_distr_channel','X_vehicle_type','X_ttm_claims','X_policy_count','X_risk_code',"X_vehicle_age",
                           'X_policy_tenure','X_policy_premium','X_prev_renewal_perc','X_upcoming_premium','X_cur_renewal_perc','X_year','1-Z', 'U','fold_number','churn_prediction','Z','prob_acceptance']

['id',
 'X_age',
 'X_bonus_malus_rating',
 'X_distr_channel',
 'X_vehicle_type',
 'X_ttm_claims',
 'X_policy_count',
 'X_risk_code',
 'X_vehicle_age',
 'X_policy_tenure',
 'X_policy_premium',
 'X_prev_renewal_perc',
 'X_upcoming_premium',
 'X_cur_renewal_perc',
 'X_year',
 '1-Z',
 'U',
 'fold_number',
 'churn_prediction',
 'Z',
 'prob_acceptance']

In [347]:
df_middle_class.loc[:, ['id', 'X_age','X_bonus_malus_rating','X_distr_channel','X_vehicle_type','X_ttm_claims','X_policy_count','X_risk_code','X_vehicle_age',
                           'X_policy_tenure','X_policy_premium', 'U','prob_acceptance','Z']].to_csv('df_acceptance_xgb_black_box.csv',sep=';')

In [348]:
df_middle.loc[:,['id', 'X_age','X_bonus_malus_rating','X_distr_channel','X_vehicle_type','X_ttm_claims','X_policy_count','X_risk_code','X_vehicle_age','X_policy_tenure',
                     'Y','U','Y_hat']]

,id,X_age,X_bonus_malus_rating,X_distr_channel,X_vehicle_type,X_ttm_claims,X_policy_count,X_risk_code,X_vehicle_age,X_policy_tenure,Y,U,Y_hat
7,5,93.0,45.0,5,7,0,1,0,24.0,24.0,175.7725,0.123099,193.966034
16,85,78.0,45.0,5,0,0,2,0,5.0,24.0,180.8686,0.107047,162.414307
24,93,78.0,45.0,5,1,0,1,0,21.0,24.0,165.6403,0.096895,168.083298
32,119,49.0,45.0,5,1,0,1,0,19.0,24.0,171.3347,0.097209,175.712479
118,395,49.0,45.0,2,2,0,4,0,16.0,24.0,170.0981,0.102011,210.380798
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3370174,5732129,68.0,45.0,4,2,0,1,2,9.0,9.0,345.4078,0.062686,267.912628
3370323,5732370,81.0,45.0,4,4,0,5,0,20.0,10.0,234.6113,0.087288,233.719376
3370329,5732373,58.0,45.0,4,7,0,3,0,30.0,8.0,174.1869,0.087051,193.689865
3370407,5732452,89.0,45.0,4,1,0,2,0,10.0,23.0,238.8796,0.087493,184.481094


In [349]:
df_middle['U'] = 1 + df_middle['U']

In [352]:
df_middle.columns

Index(['id', 'X_age', 'X_bonus_malus_rating', 'X_distr_channel',
       'X_vehicle_type', 'X_ttm_claims', 'X_policy_count', 'X_risk_code',
       'X_vehicle_age', 'X_policy_tenure', 'Y', 'X_prev_renewal_perc',
       'X_upcoming_premium', 'X_cur_renewal_perc', 'X_year', '1-Z', 'U',
       'fold_number', 'Y_hat', 'Y_residual', 'Y_absolute_error'],
      dtype='object')

In [353]:
df_middle.loc[:,['id', 'X_age','X_bonus_malus_rating','X_distr_channel','X_vehicle_type','X_ttm_claims','X_policy_count','X_risk_code','X_vehicle_age','X_policy_tenure',
                     'Y','U','Y_hat']].to_csv('df_exp_financial_loss_xgb_black_box.csv',sep=';')

In [354]:
from pygam import LinearGAM, s
from pygam.datasets import toy_interaction

X, y = toy_interaction(return_X_y=True)

gam = LinearGAM(s(0, by=1)).fit(X, y)
gam.summary()

LinearGAM                                                                                                 
=============================================== ==========================================================
Distribution:                        NormalDist Effective DoF:                                     20.8514
Link Function:                     IdentityLink Log Likelihood:                                 44131.6223
Number of Samples:                        50000 AIC:                                           -88219.5417
                                                AICc:                                          -88219.5218
                                                GCV:                                                  0.01
                                                Scale:                                              0.1001
                                                Pseudo R-Squared:                                   0.9976
Feature Function                  Lam

C:\Users\malosett\AppData\Local\Temp\ipykernel_22496\427681795.py:7: UserWarning: KNOWN BUG: p-values computed in this summary are likely much smaller than they should be. 
 
Please do not make inferences based on these values! 

Collaborate on a solution, and stay up to date at: 
github.com/dswah/pyGAM/issues/163 

  gam.summary()


In [355]:
!pip install pygam

Defaulting to user installation because normal site-packages is not writeable


In [357]:
df_sub.columns

Index(['encoded_policy_number', 'X_age', 'X_bonus_malus_rating',
       'X_distr_channel', 'X_vehicle_type', 'X_ttm_claims', 'X_policy_count',
       'X_risk_code', 'X_vehicle_age', 'X_policy_tenure', 'X_policy_premium',
       'X_prev_renewal_perc', 'X_upcoming_premium', 'X_cur_renewal_perc',
       'year', 'is_churn'],
      dtype='object')

## Section 10 — GLM & GAM Dataset Preparation

Three additional datasets are created for the interpretable models, all using the same `filter_middle_advanced` thresholds:

| Dataset | Used by |
|---|---|
| `df_middle_glm` | GLM (Logistic Regression) churn classifier |
| `df_middle_gam` | GAM (LogisticGAM) churn classifier |
| `df_middle_reg` | (supporting dataset, currently superseded by `df_middle_reg_linear`) |

For GAM and GLM, categorical columns are **label-encoded** so that the preprocessors within each pipeline start from the same integer representation.  
`U` is computed and added to each dataset before training.

In [358]:
df_middle_glm = filter_middle_advanced(
    df_sub,
    numeric_filters={
        'X_age': {'multiplier': 1.5, 'method': 'iqr'},
        'X_policy_premium': {'multiplier': 2.0, 'method': 'iqr'},
        'X_upcoming_premium':{'multiplier':2.0,'method':'iqr'},
        'X_vehicle_age': {'method': 'percentile', 'lower_q': 0.05, 'upper_q': 0.95},
        'X_cur_renewal_perc':{'method': 'percentile', 'lower_q': 0.05, 'upper_q': 0.95}
    },
    categorical_filters={
        'X_distr_channel': {'max_categories':12},
        'X_risk_code': {'max_categories': 6},
        'X_vehicle_type': {'max_categories': 8}  # explicit list
    },
    keep_id_cols=['encoded_policy_number']
)

X_age: [8.50, 100.50]
X_policy_premium: [-14.06, 444.14]
X_upcoming_premium: [-15.07, 480.13]
X_vehicle_age: [4.00, 31.00]
X_cur_renewal_perc: [-0.02, 0.11]
X_distr_channel: Kept 12 categories
X_risk_code: Kept 6 categories
X_vehicle_type: Kept 8 categories

Total: 463400 → 194373 rows


In [359]:
df_middle_glm['U'] = (df_middle_glm['X_upcoming_premium'] - df_middle_glm['X_policy_premium']) / df_middle_glm['X_policy_premium']

In [360]:
df_middle_gam = filter_middle_advanced(
    df_sub,
    numeric_filters={
        'X_age': {'multiplier': 1.5, 'method': 'iqr'},
        'X_policy_premium': {'multiplier': 2.0, 'method': 'iqr'},
        'X_upcoming_premium':{'multiplier':2.0,'method':'iqr'},
        'X_vehicle_age': {'method': 'percentile', 'lower_q': 0.05, 'upper_q': 0.95},
        'X_cur_renewal_perc':{'method': 'percentile', 'lower_q': 0.05, 'upper_q': 0.95}
    },
    categorical_filters={
        'X_distr_channel': {'max_categories':12},
        'X_risk_code': {'max_categories': 6},
        'X_vehicle_type': {'max_categories': 8}  # explicit list
    },
    keep_id_cols=['encoded_policy_number']
)

X_age: [8.50, 100.50]
X_policy_premium: [-14.06, 444.14]
X_upcoming_premium: [-15.07, 480.13]
X_vehicle_age: [4.00, 31.00]
X_cur_renewal_perc: [-0.02, 0.11]
X_distr_channel: Kept 12 categories
X_risk_code: Kept 6 categories
X_vehicle_type: Kept 8 categories

Total: 463400 → 194373 rows


In [361]:
df_middle_gam.columns

Index(['encoded_policy_number', 'X_age', 'X_bonus_malus_rating',
       'X_distr_channel', 'X_vehicle_type', 'X_ttm_claims', 'X_policy_count',
       'X_risk_code', 'X_vehicle_age', 'X_policy_tenure', 'X_policy_premium',
       'X_prev_renewal_perc', 'X_upcoming_premium', 'X_cur_renewal_perc',
       'year', 'is_churn'],
      dtype='object')

In [362]:
df_middle_gam['U'] = (df_middle_gam['X_upcoming_premium'] - df_middle_gam['X_policy_premium']) / df_middle_gam['X_policy_premium']

In [363]:
df_middle_reg = filter_middle_advanced(
    df_sub,
    numeric_filters={
        'X_age': {'multiplier': 1.5, 'method': 'iqr'},
        'X_policy_premium': {'multiplier': 2.0, 'method': 'iqr'},
        'X_upcoming_premium':{'multiplier':2.0,'method':'iqr'},
        'X_vehicle_age': {'method': 'percentile', 'lower_q': 0.05, 'upper_q': 0.95},
        'X_cur_renewal_perc':{'method': 'percentile', 'lower_q': 0.05, 'upper_q': 0.95}
    },
    categorical_filters={
        'X_distr_channel': {'max_categories':12},
        'X_risk_code': {'max_categories': 6},
        'X_vehicle_type': {'max_categories': 8}  # explicit list
    },
    keep_id_cols=['encoded_policy_number']
)

X_age: [8.50, 100.50]
X_policy_premium: [-14.06, 444.14]
X_upcoming_premium: [-15.07, 480.13]
X_vehicle_age: [4.00, 31.00]
X_cur_renewal_perc: [-0.02, 0.11]
X_distr_channel: Kept 12 categories
X_risk_code: Kept 6 categories
X_vehicle_type: Kept 8 categories

Total: 463400 → 194373 rows


In [364]:
df_middle_reg['U'] = (df_middle_reg['X_upcoming_premium'] - df_middle_reg['X_policy_premium']) / df_middle_reg['X_policy_premium']

In [366]:
from sklearn.preprocessing import LabelEncoder
for cat in cat_cols:
    le = LabelEncoder()
    le.fit(df_middle_glm[cat])
    df_middle_glm[cat] = le.transform(df_middle_glm[cat])
    df_middle_gam[cat] = le.transform(df_middle_gam[cat])
    df_middle_reg[cat] = le.transform(df_middle_reg[cat])


## Section 11 — GLM Churn Cross-Validation (Logistic Regression)

### `ModelGLM`
A **sklearn `Pipeline`** wrapping:
1. `ColumnTransformer` — median imputation + standard scaling for numerics; mode imputation + one-hot encoding for categoricals
2. `LogisticRegression` (solver: `lbfgs`, max_iter: 1000)

**Tuning:** Optuna over the regularisation parameter `C` (log-uniform in `[0.001, 100]`), maximising ROC-AUC.

### `cross_validate_glm_with_oof`
- 5-fold outer CV with an inner 80/20 tuning split per fold  
- Outputs OOF churn probabilities (`oof_prediction_glm`) and the fold assignment

**Why GLM?**  
Provides an interpretable, linear-in-log-odds alternative to XGBoost.  
Suitable when a regulator requires explainability of acceptance probability estimates.

In [367]:
from sklearn import metrics
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
import optuna
from optuna.trial import Trial


class ModelGLM():
    """
    GLM-like classifier implemented with sklearn LogisticRegression.
    This avoids statsmodels exog shape mismatches across folds.
    """

    def __init__(self):
        self.categorical_cols = []
        self.numeric_cols = []

    def _build_model(self, C: float) -> Pipeline:
        numeric_transformer = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ])

        categorical_transformer = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ])

        preprocessor = ColumnTransformer(
            transformers=[
                ("num", numeric_transformer, self.numeric_cols),
                ("cat", categorical_transformer, self.categorical_cols),
            ],
            remainder="drop",
        )

        clf = LogisticRegression(
            C=C,
            max_iter=1000,
            solver="lbfgs",
            random_state=123,
        )

        model = Pipeline(steps=[
            ("preprocessor", preprocessor),
            ("classifier", clf),
        ])
        return model

    def objective(self, trial: Trial, X_train: pd.DataFrame, y_train: pd.Series,
                  X_val: pd.DataFrame, y_val: pd.Series) -> float:
        """Optuna objective for logistic regression."""
        C = trial.suggest_float("C", 1e-3, 100.0, log=True)

        # Infer schema from training fold only.
        self.categorical_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
        self.numeric_cols = [c for c in X_train.columns if c not in self.categorical_cols]

        try:
            model = self._build_model(C=C)
            model.fit(X_train, y_train)
            y_score = model.predict_proba(X_val)[:, 1]
            y_score = np.clip(y_score, 1e-10, 1 - 1e-10)
            roc_auc = metrics.roc_auc_score(y_true=y_val, y_score=y_score)
            return roc_auc
        except Exception as e:
            print(f"Error in LogisticRegression fitting: {e}")
            return 0.5

    def tuning(self, X_train: pd.DataFrame, y_train: pd.Series,
               X_val: pd.DataFrame, y_val: pd.Series, n_trials: int) -> dict:
        """Hyperparameter tuning using Optuna."""
        study = optuna.create_study(study_name="LogisticRegression", direction="maximize")
        study.optimize(
            lambda trial: self.objective(trial, X_train, y_train, X_val, y_val),
            n_trials=n_trials,
            show_progress_bar=True,
        )
        return study.best_params

    def train(self, X_train: pd.DataFrame, y_train: pd.Series,
              X_val: pd.DataFrame, y_val: pd.Series, best_params: dict):
        """Train final logistic regression with best parameters."""
        C = best_params.get("C", 1.0)
        self.categorical_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
        self.numeric_cols = [c for c in X_train.columns if c not in self.categorical_cols]

        model = self._build_model(C=C)
        model.fit(X_train, y_train)
        return model


def cross_validate_glm_with_oof(X, y, n_splits=5, n_trials=20):
    """Cross-validation with OOF predictions for Logistic Regression GLM."""
    kfold = KFold(n_splits=n_splits, shuffle=True, random_state=123)

    fold_numbers = np.zeros(len(X))
    oof_predictions_proba = np.zeros(len(X))
    oof_predictions_class = np.zeros(len(X))

    fold_results = []
    best_params_per_fold = []

    for fold_idx, (train_idx, val_idx) in enumerate(kfold.split(X)):
        print(f"\n{'='*70}")
        print(f"Fold {fold_idx + 1}/{n_splits}")
        print(f"{'='*70}")

        fold_numbers[val_idx] = fold_idx + 1

        X_train_fold = X.iloc[train_idx].reset_index(drop=True)
        X_val_fold = X.iloc[val_idx].reset_index(drop=True)
        y_train_fold = y.iloc[train_idx].reset_index(drop=True)
        y_val_fold = y.iloc[val_idx].reset_index(drop=True)

        # Split for tuning
        X_train_tune, X_val_tune, y_train_tune, y_val_tune = train_test_split(
            X_train_fold, y_train_fold, test_size=0.2, random_state=123, stratify=y_train_fold
        )

        # Tune
        model_glm = ModelGLM()
        best_params = model_glm.tuning(
            X_train_tune, y_train_tune,
            X_val_tune, y_val_tune,
            n_trials=n_trials,
        )

        print(f"Best params: {best_params}")
        best_params_per_fold.append(best_params)

        # Train
        final_model = model_glm.train(
            X_train_fold, y_train_fold,
            X_val_fold, y_val_fold,
            best_params,
        )

        # Predict
        y_pred_proba = final_model.predict_proba(X_val_fold)[:, 1]
        y_pred_proba = np.clip(y_pred_proba, 0, 1)
        y_pred_class = (y_pred_proba > 0.5).astype(int)

        oof_predictions_proba[val_idx] = y_pred_proba
        oof_predictions_class[val_idx] = y_pred_class

        # Metrics
        roc_auc = metrics.roc_auc_score(y_val_fold, y_pred_proba)
        accuracy = metrics.accuracy_score(y_val_fold, y_pred_class)

        fold_results.append({
            "fold": fold_idx + 1,
            "roc_auc": roc_auc,
            "accuracy": accuracy,
            "n_train": len(train_idx),
            "n_val": len(val_idx),
        })

        print(f"Fold {fold_idx + 1} ROC-AUC: {roc_auc:.4f}")
        print(f"Fold {fold_idx + 1} Accuracy: {accuracy:.4f}")

    results_df = pd.DataFrame(fold_results)

    print(f"\n{'='*70}")
    print("Cross-Validation Summary")
    print(f"{'='*70}")
    print(f"Mean ROC-AUC: {results_df['roc_auc'].mean():.4f} (+/- {results_df['roc_auc'].std():.4f})")
    print(f"Mean Accuracy: {results_df['accuracy'].mean():.4f} (+/- {results_df['accuracy'].std():.4f})")

    overall_roc_auc = metrics.roc_auc_score(y, oof_predictions_proba)
    print(f"\nOverall OOF ROC-AUC: {overall_roc_auc:.4f}")

    return fold_numbers, oof_predictions_proba, results_df, best_params_per_fold


# Usage
X = df_middle_glm.loc[:, model_features_class]
y = df_middle_glm['is_churn']

fold_numbers_glm, oof_pred_glm, results_glm, params_glm = cross_validate_glm_with_oof(
    X, y, n_splits=5, n_trials=20
)

df_middle_glm['fold_number_glm'] = fold_numbers_glm.astype(int)
df_middle_glm['oof_prediction_glm'] = oof_pred_glm

[I 2026-03-31 10:08:17,780] A new study created in memory with name: LogisticRegression



Fold 1/5


  0%|          | 0/20 [00:00<?, ?it/s]

[I 2026-03-31 10:08:18,524] Trial 0 finished with value: 0.6511430838662619 and parameters: {'C': 1.2391445655399789}. Best is trial 0 with value: 0.6511430838662619.
[I 2026-03-31 10:08:19,145] Trial 1 finished with value: 0.6511532317480689 and parameters: {'C': 0.059422929543679635}. Best is trial 1 with value: 0.6511532317480689.
[I 2026-03-31 10:08:19,403] Trial 2 finished with value: 0.6511425567035705 and parameters: {'C': 94.5322401931766}. Best is trial 1 with value: 0.6511532317480689.
[I 2026-03-31 10:08:19,659] Trial 3 finished with value: 0.651498776754457 and parameters: {'C': 0.0013663406966000849}. Best is trial 3 with value: 0.651498776754457.
[I 2026-03-31 10:08:19,938] Trial 4 finished with value: 0.651207448403318 and parameters: {'C': 0.009142441029275604}. Best is trial 3 with value: 0.651498776754457.
[I 2026-03-31 10:08:20,182] Trial 5 finished with value: 0.6512052586506004 and parameters: {'C': 0.009573933926979491}. Best is trial 3 with value: 0.6514987767544

[I 2026-03-31 10:08:23,511] A new study created in memory with name: LogisticRegression


...

[I 2026-03-31 10:08:33,608] Trial 0 finished with value: 0.6437469872043566 and parameters: {'C': 20.529952161463886}. Best is trial 0 with value: 0.6437469872043566.
[I 2026-03-31 10:08:33,843] Trial 1 finished with value: 0.6437474165358815 and parameters: {'C': 2.4373876989871532}. Best is trial 1 with value: 0.6437474165358815.
[I 2026-03-31 10:08:34,152] Trial 2 finished with value: 0.6437469974265357 and parameters: {'C': 19.096089752404673}. Best is trial 1 with value: 0.6437474165358815.
[I 2026-03-31 10:08:34,378] Trial 3 finished with value: 0.6437469258712816 and parameters: {'C': 90.1469089833352}. Best is trial 1 with value: 0.6437474165358815.
[I 2026-03-31 10:08:34,619] Trial 4 finished with value: 0.6437475187576732 and parameters: {'C': 1.7963411087257362}. Best is trial 4 with value: 0.6437475187576732.
[I 2026-03-31 10:08:34,880] Trial 5 finished with value: 0.6438188491238964 and parameters: {'C': 0.0069960703018583075}. Best is trial 5 with value: 0.643818849123896

[I 2026-03-31 10:08:38,482] A new study created in memory with name: LogisticRegression


Fold 4 ROC-AUC: 0.6393
Fold 4 Accuracy: 0.8809

Fold 5/5


  0%|          | 0/20 [00:00<?, ?it/s]

[I 2026-03-31 10:08:38,799] Trial 0 finished with value: 0.6459974896294033 and parameters: {'C': 0.12026725402462285}. Best is trial 0 with value: 0.6459974896294033.
[I 2026-03-31 10:08:39,061] Trial 1 finished with value: 0.6460013539850042 and parameters: {'C': 0.02217557195366564}. Best is trial 1 with value: 0.6460013539850042.
[I 2026-03-31 10:08:39,298] Trial 2 finished with value: 0.6460012424156824 and parameters: {'C': 0.029531814551313474}. Best is trial 1 with value: 0.6460013539850042.
[I 2026-03-31 10:08:39,634] Trial 3 finished with value: 0.6460004411450985 and parameters: {'C': 0.03728206570382278}. Best is trial 1 with value: 0.6460013539850042.
[I 2026-03-31 10:08:39,834] Trial 4 finished with value: 0.6460222783041759 and parameters: {'C': 0.005893368931531933}. Best is trial 4 with value: 0.6460222783041759.
[I 2026-03-31 10:08:40,037] Trial 5 finished with value: 0.6459967289294819 and parameters: {'C': 0.1524358559570441}. Best is trial 4 with value: 0.646022278

In [369]:
df_middle_glm.head()


,encoded_policy_number,X_age,X_bonus_malus_rating,X_distr_channel,X_vehicle_type,X_ttm_claims,X_policy_count,X_risk_code,X_vehicle_age,X_policy_tenure,X_policy_premium,X_prev_renewal_perc,X_upcoming_premium,X_cur_renewal_perc,year,is_churn,U,fold_number_glm,oof_prediction_glm
7,5,93.0,45.0,5,7,0,1,0,24.0,24.0,175.7725,0.003637,197.41,0.098601,7,False,0.123099,5,0.133983
16,85,78.0,45.0,5,0,0,2,0,5.0,24.0,180.8686,0.003824,200.23,0.082852,7,False,0.107047,4,0.049107
24,93,78.0,45.0,5,1,0,1,0,21.0,24.0,165.6403,0.003731,181.69,0.072973,7,False,0.096895,1,0.088342
32,119,49.0,45.0,5,1,0,1,0,19.0,24.0,171.3347,-0.005503,187.99,0.073266,7,False,0.097209,5,0.100304
118,395,49.0,45.0,2,2,0,4,0,16.0,24.0,170.0981,-0.005223,187.45,0.077961,7,False,0.102011,5,0.067991


## Section 12 — GLM Output: Acceptance Probability

Post-CV steps for `df_middle_glm`:

1. Compute `Z = 1 − is_churn` (binary acceptance)  
2. Compute `prob_acceptance = 1 − oof_prediction_glm`  
3. Adjust `U = 1 + U` (uplift factor format)  
4. Rename columns to canonical business names  

**Exported to:** `df_acceptance_linear_model_black_box.csv`

This file mirrors the XGBoost acceptance output but with GLM-derived acceptance probabilities, allowing the optimisation module to compare model choices.

In [370]:
df_middle_glm['Z'] = 1 - df_middle_glm['is_churn'].astype(int)
df_middle_glm['prob_acceptance'] = 1 - df_middle_glm['oof_prediction_glm']

In [373]:
df_middle_glm['U'] = 1 + df_middle_glm['U']

In [371]:
df_middle_glm.columns = ['id','X_age','X_bonus_malus_rating','X_distr_channel','X_vehicle_type','X_ttm_claims','X_policy_count','X_risk_code','X_vehicle_age','X_policy_tenure',
                         'X_policy_premium','X_prev_renewal_perc','X_upcoming_premium','X_cur_renewal_perc','X_year','1-Z','U','fold_number_glm','churn_prediction','Z','prob_acceptance']

In [374]:
df_middle_glm.columns

Index(['id', 'X_age', 'X_bonus_malus_rating', 'X_distr_channel',
       'X_vehicle_type', 'X_ttm_claims', 'X_policy_count', 'X_risk_code',
       'X_vehicle_age', 'X_policy_tenure', 'X_policy_premium',
       'X_prev_renewal_perc', 'X_upcoming_premium', 'X_cur_renewal_perc',
       'X_year', '1-Z', 'U', 'fold_number_glm', 'churn_prediction', 'Z',
       'prob_acceptance'],
      dtype='object')

In [375]:
df_middle_glm.to_csv('df_acceptance_linear_model_black_box.csv',sep=';')

In [127]:
df_middle_gam.columns

Index(['encoded_policy_number', 'age', 'bonus_malus_rating', 'distr_channel',
       'vehicle_type', 'ttm_claims', 'policy_count', 'risk_code',
       'vehicle_age', 'policy_tenure', 'policy_premium', 'prev_renewal_perc',
       'upcoming_premium', 'cur_renewal_perc', 'year', 'is_churn', 'U'],
      dtype='object')

## Section 13 — GAM Churn Cross-Validation (LogisticGAM)

### `ModelGAM` (memory-safe version)
A **pyGAM `LogisticGAM`** that uses:
- **Spline terms** (`s()`) for continuous features — captures non-linear relationships
- **Factor terms** (`f()`) for categorical features — discrete category effects

Key design decisions:
- `_prepare_data` — category mappings are fitted on training data only (prevents leakage)
- Progressively tries `n_splines = [8, 6, 4]` if fitting fails due to memory pressure
- `max_train_samples = 35,000` — stratified subsampling is applied to avoid OOM on large folds

### `cross_validate_gam_with_oof`
- 5-fold outer CV (no inner Optuna tuning; `n_splines` and `lam` are fixed hyperparameters)
- `_stratified_sample` ensures class balance is preserved after subsampling

**Why GAM?**  
GAMs are semi-parametric: they capture non-linear covariate effects while remaining interpretable via partial dependence plots, making them a strong regulatory-friendly alternative.

In [128]:
from sklearn import metrics
import pandas as pd
import numpy as np
from pygam import LogisticGAM, s, f
from sklearn.model_selection import KFold


class ModelGAM:
    """Memory-safe GAM classifier for churn prediction."""

    def __init__(self):
        self.numeric_cols = []
        self.categorical_cols = []
        self.feature_order = []
        self.category_maps = {}
        self.category_default_code = {}

    def _prepare_data(self, X: pd.DataFrame, fit: bool = False) -> pd.DataFrame:
        """Prepare data for GAM using stable category mappings and fixed column order."""
        X_prep = X.copy()

        if fit:
            self.numeric_cols = X_prep.select_dtypes(include=[np.number]).columns.tolist()
            self.categorical_cols = X_prep.select_dtypes(include=["object", "category"]).columns.tolist()
            self.feature_order = self.numeric_cols + self.categorical_cols

            self.category_maps = {}
            self.category_default_code = {}
            for col in self.categorical_cols:
                cats = sorted(X_prep[col].astype(str).dropna().unique().tolist())
                mapping = {cat: idx for idx, cat in enumerate(cats)}
                self.category_maps[col] = mapping
                self.category_default_code[col] = len(mapping)

        for col in self.categorical_cols:
            mapping = self.category_maps[col]
            default_code = self.category_default_code[col]
            X_prep[col] = X_prep[col].astype(str).map(lambda x: mapping.get(x, default_code))

        if len(self.numeric_cols) > 0:
            X_prep[self.numeric_cols] = X_prep[self.numeric_cols].apply(
                lambda s_col: s_col.fillna(s_col.median())
            )

        X_prep = X_prep[self.feature_order]
        return X_prep

    def _build_formula(self, n_features: int, n_numeric: int, n_splines: int = 8, lam: float = 0.8):
        if n_features == 0:
            raise ValueError("No features found for GAM model.")

        terms = None

        for i in range(n_numeric):
            term = s(i, n_splines=n_splines, lam=lam)
            terms = term if terms is None else terms + term

        for i in range(n_numeric, n_features):
            term = f(i, lam=lam)
            terms = term if terms is None else terms + term

        return terms

    def train(
        self,
        X_train: pd.DataFrame,
        y_train: pd.Series,
        X_val: pd.DataFrame,
        y_val: pd.Series,
        n_splines: int = 8,
        lam: float = 0.8,
    ):
        X_train_prep = self._prepare_data(X_train, fit=True)
        _ = self._prepare_data(X_val, fit=False)

        n_features = X_train_prep.shape[1]
        n_numeric = len(self.numeric_cols)

        # Try progressively simpler models if memory pressure occurs.
        candidate_splines = [n_splines, min(6, n_splines), 4]

        last_error = None
        for spl in candidate_splines:
            try:
                terms = self._build_formula(n_features, n_numeric, n_splines=spl, lam=lam)
                gam = LogisticGAM(terms=terms, max_iter=100)
                gam.fit(X_train_prep.values.astype(np.float32), y_train.values)
                return gam
            except Exception as e:
                last_error = e
                print(f"GAM fit failed with n_splines={spl}: {e}")

        raise RuntimeError(f"All GAM fits failed. Last error: {last_error}")


def _stratified_sample(X: pd.DataFrame, y: pd.Series, max_samples: int, seed: int):
    """Return stratified sample indices for memory-safe GAM fitting."""
    if len(X) <= max_samples:
        return X.reset_index(drop=True), y.reset_index(drop=True)

    rng = np.random.default_rng(seed)
    y_arr = y.values

    pos_idx = np.where(y_arr == 1)[0]
    neg_idx = np.where(y_arr == 0)[0]

    pos_ratio = len(pos_idx) / len(y_arr)
    n_pos = int(max_samples * pos_ratio)
    n_neg = max_samples - n_pos

    n_pos = min(n_pos, len(pos_idx))
    n_neg = min(n_neg, len(neg_idx))

    sampled_pos = rng.choice(pos_idx, size=n_pos, replace=False) if n_pos > 0 else np.array([], dtype=int)
    sampled_neg = rng.choice(neg_idx, size=n_neg, replace=False) if n_neg > 0 else np.array([], dtype=int)

    sample_idx = np.concatenate([sampled_pos, sampled_neg])
    rng.shuffle(sample_idx)

    return X.iloc[sample_idx].reset_index(drop=True), y.iloc[sample_idx].reset_index(drop=True)


def cross_validate_gam_with_oof(
    X,
    y,
    n_splits=5,
    n_splines=8,
    lam=0.8,
    max_train_samples=35000,
):
    """Cross-validation for GAM with OOF predictions and RAM-safe training."""
    kfold = KFold(n_splits=n_splits, shuffle=True, random_state=123)

    fold_numbers = np.zeros(len(X))
    oof_predictions_proba = np.zeros(len(X))
    oof_predictions_class = np.zeros(len(X))

    fold_results = []

    for fold_idx, (train_idx, val_idx) in enumerate(kfold.split(X)):
        print(f"\n{'=' * 70}")
        print(f"Fold {fold_idx + 1}/{n_splits}")
        print(f"{'=' * 70}")

        fold_numbers[val_idx] = fold_idx + 1

        X_train_fold = X.iloc[train_idx].reset_index(drop=True)
        X_val_fold = X.iloc[val_idx].reset_index(drop=True)
        y_train_fold = y.iloc[train_idx].reset_index(drop=True)
        y_val_fold = y.iloc[val_idx].reset_index(drop=True)

        X_train_fit, y_train_fit = _stratified_sample(
            X_train_fold,
            y_train_fold,
            max_samples=max_train_samples,
            seed=123 + fold_idx,
        )

        print(f"Training rows used: {len(X_train_fit):,} / {len(X_train_fold):,}")

        model_gam = ModelGAM()
        try:
            gam_model = model_gam.train(
                X_train_fit,
                y_train_fit,
                X_val_fold,
                y_val_fold,
                n_splines=n_splines,
                lam=lam,
            )

            X_val_prep = model_gam._prepare_data(X_val_fold, fit=False)
            y_pred_proba = gam_model.predict_proba(X_val_prep.values.astype(np.float32))
            y_pred_class = (y_pred_proba > 0.5).astype(int)

            oof_predictions_proba[val_idx] = y_pred_proba
            oof_predictions_class[val_idx] = y_pred_class

            roc_auc = metrics.roc_auc_score(y_val_fold, y_pred_proba)
            accuracy = metrics.accuracy_score(y_val_fold, y_pred_class)

            fold_results.append(
                {
                    "fold": fold_idx + 1,
                    "roc_auc": roc_auc,
                    "accuracy": accuracy,
                    "n_train_used": len(X_train_fit),
                    "n_train_full": len(train_idx),
                    "n_val": len(val_idx),
                }
            )

            print(f"Fold {fold_idx + 1} ROC-AUC: {roc_auc:.4f}")
            print(f"Fold {fold_idx + 1} Accuracy: {accuracy:.4f}")

        except Exception as e:
            print(f"Error in fold {fold_idx + 1}: {e}")
            oof_predictions_proba[val_idx] = 0.5
            oof_predictions_class[val_idx] = 0

    results_df = pd.DataFrame(fold_results)

    print(f"\n{'=' * 70}")
    print("Cross-Validation Summary")
    print(f"{'=' * 70}")
    if len(results_df) > 0:
        print(
            f"Mean ROC-AUC: {results_df['roc_auc'].mean():.4f} "
            f"(+/- {results_df['roc_auc'].std():.4f})"
        )
        print(
            f"Mean Accuracy: {results_df['accuracy'].mean():.4f} "
            f"(+/- {results_df['accuracy'].std():.4f})"
        )

        overall_roc_auc = metrics.roc_auc_score(y, oof_predictions_proba)
        print(f"\nOverall OOF ROC-AUC: {overall_roc_auc:.4f}")

    return fold_numbers, oof_predictions_proba, results_df


# Usage with your data
X = df_middle_gam.loc[:, model_features_class]
y = df_middle_gam["is_churn"]

print("Running GAM Cross-Validation...")
fold_numbers_gam, oof_pred_gam, results_gam = cross_validate_gam_with_oof(
    X,
    y,
    n_splits=5,
    n_splines=8,
    lam=0.8,
    max_train_samples=35000,
)

df_middle_gam["fold_number_gam"] = fold_numbers_gam.astype(int)
df_middle_gam["oof_prediction_gam"] = oof_pred_gam

Running GAM Cross-Validation...

Fold 1/5
Training rows used: 35,000 / 155,498
Fold 1 ROC-AUC: 0.6622
Fold 1 Accuracy: 0.8854

Fold 2/5
Training rows used: 35,000 / 155,498
Fold 2 ROC-AUC: 0.6599
Fold 2 Accuracy: 0.8858

Fold 3/5
Training rows used: 35,000 / 155,498
Fold 3 ROC-AUC: 0.6520
Fold 3 Accuracy: 0.8870

Fold 4/5
Training rows used: 35,000 / 155,499
Fold 4 ROC-AUC: 0.6575
Fold 4 Accuracy: 0.8810

Fold 5/5
Training rows used: 35,000 / 155,499
Fold 5 ROC-AUC: 0.6640
Fold 5 Accuracy: 0.8851

Cross-Validation Summary
Mean ROC-AUC: 0.6591 (+/- 0.0047)
Mean Accuracy: 0.8849 (+/- 0.0023)

Overall OOF ROC-AUC: 0.6590


In [129]:
import pygam

In [68]:
# Update narwhals to the latest version
!pip install --upgrade narwhals

# Or if that doesn't work, try reinstalling both packages
!pip install --upgrade glum narwhals


Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


In [130]:
df_middle_gam.columns

Index(['encoded_policy_number', 'age', 'bonus_malus_rating', 'distr_channel',
       'vehicle_type', 'ttm_claims', 'policy_count', 'risk_code',
       'vehicle_age', 'policy_tenure', 'policy_premium', 'prev_renewal_perc',
       'upcoming_premium', 'cur_renewal_perc', 'year', 'is_churn', 'U',
       'fold_number_gam', 'oof_prediction_gam'],
      dtype='object')

In [132]:
df_middle_gam.columns = ['id','X_age','X_bonus_malus_rating','X_distr_channel','X_vehicle_type','X_ttm_claims',
                         'X_policy_count','X_risk_code','X_vehicle_age','X_policy_tenure','X_policy_premium',
                         'X_prev_renewal_perc','X_upcoming_premium','X_cur_renewal_perc','X_year','1-Z','U',
                         'fold_number_gam','churn_prediction']

## Section 14 — GAM Output: Acceptance Probability

Post-CV steps for `df_middle_gam`:

1. Compute `Z = 1 − churn_prediction` (binary acceptance, where `churn_prediction = oof_prediction_gam`)  
2. Compute `prob_acceptance = 1 − churn_prediction`  
3. Rename columns to canonical names  

**Exported to:** `df_acceptance_gam_model_black_box.csv`

Same schema as the XGBoost and GLM acceptance files — the optimiser can swap between models with no interface change.

In [133]:
df_middle_gam['Z'] = 1 - df_middle_gam['1-Z'].astype(int)
df_middle_gam['prob_acceptance'] = 1 - df_middle_gam['churn_prediction']

In [134]:
df_middle_gam.columns

Index(['id', 'X_age', 'X_bonus_malus_rating', 'X_distr_channel',
       'X_vehicle_type', 'X_ttm_claims', 'X_policy_count', 'X_risk_code',
       'X_vehicle_age', 'X_policy_tenure', 'X_policy_premium',
       'X_prev_renewal_perc', 'X_upcoming_premium', 'X_cur_renewal_perc',
       'X_year', '1-Z', 'U', 'fold_number_gam', 'churn_prediction', 'Z',
       'prob_acceptance'],
      dtype='object')

In [135]:
df_middle_gam.loc[:, ['id', 'X_age','X_bonus_malus_rating','X_distr_channel','X_vehicle_type','X_ttm_claims','X_policy_count','X_risk_code','X_vehicle_age',
                           'X_policy_tenure','X_policy_premium', 'U','prob_acceptance','Z']].to_csv('df_acceptance_gam_model_black_box.csv',sep=';')

In [136]:
df_middle_gam.columns

Index(['id', 'X_age', 'X_bonus_malus_rating', 'X_distr_channel',
       'X_vehicle_type', 'X_ttm_claims', 'X_policy_count', 'X_risk_code',
       'X_vehicle_age', 'X_policy_tenure', 'X_policy_premium',
       'X_prev_renewal_perc', 'X_upcoming_premium', 'X_cur_renewal_perc',
       'X_year', '1-Z', 'U', 'fold_number_gam', 'churn_prediction', 'Z',
       'prob_acceptance'],
      dtype='object')

In [137]:
from sklearn import metrics
import pandas as pd
import numpy as np
from pygam import LogisticGAM, s, f, te
from sklearn.model_selection import KFold, train_test_split
import optuna
from optuna.trial import Trial

class ModelGAM():
    """
    GAM Classifier for Churn Prediction
    GAMs are interpretable and can capture non-linear relationships
    """
    
    def objective(self, trial: Trial, X_train: pd.DataFrame, y_train: pd.Series, 
                  X_val: pd.DataFrame, y_val: pd.Series) -> float:
        """
        Optuna objective for hyperparameter tuning
        """
        # Suggest hyperparameters
        n_splines = trial.suggest_int("n_splines", 10, 50)
        lam = trial.suggest_float("lam", 0.01, 100, log=True)
        
        # Build GAM formula
        # Use splines for continuous features and factors for categorical
        numeric_cols = X_train.select_dtypes(include=[np.number]).columns
        categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns
        
        # Create formula
        formula_terms = []
        for i, col in enumerate(numeric_cols):
            formula_terms.append(s(i, n_splines=n_splines, lam=lam))
        
        # For categorical, use factor terms
        cat_offset = len(numeric_cols)
        for i, col in enumerate(categorical_cols):
            formula_terms.append(f(cat_offset + i, lam=lam))
        
        # Combine numeric and categorical data
        X_train_prepared = self._prepare_data(X_train)
        X_val_prepared = self._prepare_data(X_val)
        
        # Fit model
        try:
            gam_terms = formula_terms[0]
            for term in formula_terms[1:]:
                gam_terms += term
            gam = LogisticGAM(terms=gam_terms, max_iter=100)
            gam.gridsearch(X_train_prepared.values, y_train.values, progress=False)
            
            # Predict
            y_score = gam.predict_proba(X_val_prepared.values)
            roc_auc = metrics.roc_auc_score(y_true=y_val, y_score=y_score)
            
            return roc_auc
        except Exception as e:
            print(f"Error in GAM fitting: {e}")
            return 0.5  # Return baseline score on error
    
    def _prepare_data(self, X: pd.DataFrame) -> pd.DataFrame:
        """
        Prepare data for GAM
        """
        X_prep = X.copy()
        
        # Encode categorical variables as numeric
        categorical_cols = X_prep.select_dtypes(include=['object', 'category']).columns
        if len(categorical_cols) > 0:
            from sklearn.preprocessing import LabelEncoder
            for col in categorical_cols:
                le = LabelEncoder()
                X_prep[col] = le.fit_transform(X_prep[col].astype(str))
        
        return X_prep
    
    def tuning(self, X_train: pd.DataFrame, y_train: pd.Series, 
               X_val: pd.DataFrame, y_val: pd.Series, n_trials: int) -> dict:
        """
        Hyperparameter tuning using Optuna
        """
        study = optuna.create_study(study_name='GAM', direction='maximize')
        study.optimize(lambda trial: self.objective(trial, X_train, y_train, X_val, y_val), 
                      n_trials=n_trials, show_progress_bar=True)
        best_params = study.best_params
        return best_params
    
    def train(self, X_train: pd.DataFrame, y_train: pd.Series, 
              X_val: pd.DataFrame, y_val: pd.Series, best_params: dict):
        """
        Train final GAM model with best parameters
        """
        n_splines = best_params.get('n_splines', 25)
        lam = best_params.get('lam', 0.6)
        
        # Prepare data
        X_train_prep = self._prepare_data(X_train)
        X_val_prep = self._prepare_data(X_val)
        
        # Build formula
        numeric_cols = X_train_prep.select_dtypes(include=[np.number]).columns
        formula_terms = [s(i, n_splines=n_splines, lam=lam) for i in range(len(numeric_cols))]
        gam_terms = formula_terms[0]
        for term in formula_terms[1:]:
            gam_terms += term
        
        # Fit model
        gam = LogisticGAM(terms=gam_terms, max_iter=100)
        gam.gridsearch(X_train_prep.values, y_train.values, progress=False)
        
        return gam


# Cross-validation function for GAM
def cross_validate_gam_with_oof(X, y, n_splits=5, n_trials=20):
    """
    Cross-validation with OOF predictions for GAM
    """
    kfold = KFold(n_splits=n_splits, shuffle=True, random_state=123)
    
    fold_numbers = np.zeros(len(X))
    oof_predictions_proba = np.zeros(len(X))
    oof_predictions_class = np.zeros(len(X))
    
    fold_results = []
    best_params_per_fold = []
    
    for fold_idx, (train_idx, val_idx) in enumerate(kfold.split(X)):
        print(f"\n{'='*70}")
        print(f"Fold {fold_idx + 1}/{n_splits}")
        print(f"{'='*70}")
        
        fold_numbers[val_idx] = fold_idx + 1
        
        X_train_fold = X.iloc[train_idx]
        X_val_fold = X.iloc[val_idx]
        y_train_fold = y.iloc[train_idx]
        y_val_fold = y.iloc[val_idx]
        
        # Split for tuning
        X_train_tune, X_val_tune, y_train_tune, y_val_tune = train_test_split(
            X_train_fold, y_train_fold, test_size=0.2, random_state=123, stratify=y_train_fold
        )
        
        # Tune
        model_gam = ModelGAM()
        best_params = model_gam.tuning(
            X_train_tune, y_train_tune,
            X_val_tune, y_val_tune,
            n_trials=n_trials
        )
        
        print(f"Best params: {best_params}")
        best_params_per_fold.append(best_params)
        
        # Train
        final_model = model_gam.train(
            X_train_fold, y_train_fold,
            X_val_fold, y_val_fold,
            best_params
        )
        
        # Predict
        X_val_prep = model_gam._prepare_data(X_val_fold)
        y_pred_proba = final_model.predict_proba(X_val_prep.values)
        y_pred_class = (y_pred_proba > 0.5).astype(int)
        
        oof_predictions_proba[val_idx] = y_pred_proba
        oof_predictions_class[val_idx] = y_pred_class
        
        # Metrics
        roc_auc = metrics.roc_auc_score(y_val_fold, y_pred_proba)
        accuracy = metrics.accuracy_score(y_val_fold, y_pred_class)
        
        fold_results.append({
            'fold': fold_idx + 1,
            'roc_auc': roc_auc,
            'accuracy': accuracy,
            'n_train': len(train_idx),
            'n_val': len(val_idx)
        })
        
        print(f"Fold {fold_idx + 1} ROC-AUC: {roc_auc:.4f}")
        print(f"Fold {fold_idx + 1} Accuracy: {accuracy:.4f}")
    
    results_df = pd.DataFrame(fold_results)
    
    print(f"\n{'='*70}")
    print("Cross-Validation Summary")
    print(f"{'='*70}")
    print(f"Mean ROC-AUC: {results_df['roc_auc'].mean():.4f} (+/- {results_df['roc_auc'].std():.4f})")
    print(f"Mean Accuracy: {results_df['accuracy'].mean():.4f} (+/- {results_df['accuracy'].std():.4f})")
    
    overall_roc_auc = metrics.roc_auc_score(y, oof_predictions_proba)
    print(f"\nOverall OOF ROC-AUC: {overall_roc_auc:.4f}")
    
    return fold_numbers, oof_predictions_proba, results_df, best_params_per_fold

In [142]:
df_middle_glm_gam = filter_middle_advanced(
    df_sub,
    numeric_filters={
        'age': {'multiplier': 1.5, 'method': 'iqr'},
        'policy_premium': {'multiplier': 2.0, 'method': 'iqr'},
        'upcoming_premium':{'multiplier':2.0,'method':'iqr'},
        'vehicle_age': {'method': 'percentile', 'lower_q': 0.05, 'upper_q': 0.95},
        'cur_renewal_perc':{'method': 'percentile', 'lower_q': 0.05, 'upper_q': 0.95}
    },
    categorical_filters={
        'distr_channel': {'max_categories':12},
        'risk_code': {'max_categories': 6},
        'vehicle_type': {'max_categories': 8}  # explicit list
    },
    keep_id_cols=['encoded_policy_number']
)

age: [8.50, 100.50]
policy_premium: [-14.06, 444.14]
upcoming_premium: [-15.07, 480.13]
vehicle_age: [4.00, 31.00]
cur_renewal_perc: [-0.02, 0.11]
distr_channel: Kept 12 categories
risk_code: Kept 6 categories
vehicle_type: Kept 8 categories

Total: 463400 → 194373 rows


In [143]:
from sklearn.preprocessing import LabelEncoder
for cat in cat_cols:
    le = LabelEncoder()
    le.fit(df_middle_glm_gam[cat])
    df_middle_glm_gam[cat] = le.transform(df_middle_glm_gam[cat])


In [144]:
df_middle_glm_gam['U'] = (df_middle_glm_gam['upcoming_premium'] - df_middle_glm_gam['policy_premium']) / df_middle_glm_gam['policy_premium']

In [145]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression


class ModelGLM():
    """GLM replacement using sklearn Logistic Regression."""

    def __init__(self):
        self.categorical_cols = []
        self.numeric_cols = []

    def _build_model(self, C: float) -> Pipeline:
        numeric_transformer = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ])

        categorical_transformer = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ])

        preprocessor = ColumnTransformer(
            transformers=[
                ("num", numeric_transformer, self.numeric_cols),
                ("cat", categorical_transformer, self.categorical_cols),
            ],
            remainder="drop",
        )

        clf = LogisticRegression(
            C=C,
            max_iter=1000,
            solver="lbfgs",
            random_state=123,
        )

        return Pipeline(steps=[
            ("preprocessor", preprocessor),
            ("classifier", clf),
        ])

    def objective(self, trial, X_train, y_train, X_val, y_val):
        C = trial.suggest_float("C", 1e-3, 100.0, log=True)

        self.categorical_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
        self.numeric_cols = [c for c in X_train.columns if c not in self.categorical_cols]

        model = self._build_model(C=C)
        model.fit(X_train, y_train)
        y_score = model.predict_proba(X_val)[:, 1]
        return metrics.roc_auc_score(y_true=y_val, y_score=y_score)

    def tuning(self, X_train, y_train, X_val, y_val, n_trials):
        study = optuna.create_study(study_name="LogisticRegression", direction="maximize")
        study.optimize(
            lambda trial: self.objective(trial, X_train, y_train, X_val, y_val),
            n_trials=n_trials,
            show_progress_bar=True,
        )
        return study.best_params

    def train(self, X_train, y_train, X_val, y_val, best_params):
        C = best_params.get("C", 1.0)
        self.categorical_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
        self.numeric_cols = [c for c in X_train.columns if c not in self.categorical_cols]

        model = self._build_model(C=C)
        model.fit(X_train, y_train)
        return model

In [392]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold
from sklearn import metrics
import pandas as pd
import numpy as np


class ModelLinearRegressor:
    """Baseline linear regression model."""

    def train(self, X_train: pd.DataFrame, y_train: pd.Series):
        model = LinearRegression()
        model.fit(X_train, y_train)
        return model


def cross_validate_linear_regression(X, y, n_splits=5):
    """Run Linear Regression with KFold cross-validation."""
    kfold = KFold(n_splits=n_splits, shuffle=True, random_state=123)

    lin_fold_numbers = np.zeros(len(X))
    lin_oof = np.zeros(len(X))
    lin_results = []

    for fold_idx, (train_idx, val_idx) in enumerate(kfold.split(X), 1):
        print(f"\n{'='*70}")
        print(f"Fold {fold_idx}/{n_splits}")
        print(f"{'='*70}")

        X_train_fold = X.iloc[train_idx].reset_index(drop=True)
        X_val_fold = X.iloc[val_idx].reset_index(drop=True)
        y_train_fold = y.iloc[train_idx].reset_index(drop=True)
        y_val_fold = y.iloc[val_idx].reset_index(drop=True)

        lin_fold_numbers[val_idx] = fold_idx

        # Linear regression
        linear_model = ModelLinearRegressor().train(X_train_fold, y_train_fold)
        lin_pred = linear_model.predict(X_val_fold)
        lin_oof[val_idx] = lin_pred

        lin_mae = metrics.mean_absolute_error(y_val_fold, lin_pred)
        lin_rmse = np.sqrt(metrics.mean_squared_error(y_val_fold, lin_pred))
        lin_r2 = metrics.r2_score(y_val_fold, lin_pred)

        lin_results.append({
            "fold": fold_idx,
            "model": "LinearRegression",
            "mae": lin_mae,
            "rmse": lin_rmse,
            "r2": lin_r2,
            "n_train": len(train_idx),
            "n_val": len(val_idx),
        })

        print(f"LinearRegression -> MAE: {lin_mae:.4f}, RMSE: {lin_rmse:.4f}, R2: {lin_r2:.4f}")

    lin_df = pd.DataFrame(lin_results)

    print(f"\n{'='*70}")
    print("Overall OOF Metrics")
    print(f"{'='*70}")

    lin_oof_mae = metrics.mean_absolute_error(y, lin_oof)
    lin_oof_rmse = np.sqrt(metrics.mean_squared_error(y, lin_oof))
    lin_oof_r2 = metrics.r2_score(y, lin_oof)

    print(f"LinearRegression OOF -> MAE: {lin_oof_mae:.4f}, RMSE: {lin_oof_rmse:.4f}, R2: {lin_oof_r2:.4f}")

    linear_results = {
        "fold_numbers": lin_fold_numbers,
        "oof_predictions": lin_oof,
        "results_df": lin_df,
        "oof_metrics": {"mae": lin_oof_mae, "rmse": lin_oof_rmse, "r2": lin_oof_r2},
    }

    return linear_results

In [395]:
df_middle_reg_linear

,encoded_policy_number,X_age,X_bonus_malus_rating,X_distr_channel,X_vehicle_type,X_ttm_claims,X_policy_count,X_risk_code,X_vehicle_age,X_policy_tenure,X_policy_premium,X_prev_renewal_perc,X_upcoming_premium,X_cur_renewal_perc,year,is_churn
7,5,93.0,45.0,5,7,0,1,0,24.0,24.0,175.7725,0.003637,197.41,0.098601,7,False
16,85,78.0,45.0,5,0,0,2,0,5.0,24.0,180.8686,0.003824,200.23,0.082852,7,False
24,93,78.0,45.0,5,1,0,1,0,21.0,24.0,165.6403,0.003731,181.69,0.072973,7,False
32,119,49.0,45.0,5,1,0,1,0,19.0,24.0,171.3347,-0.005503,187.99,0.073266,7,False
118,395,49.0,45.0,2,2,0,4,0,16.0,24.0,170.0981,-0.005223,187.45,0.077961,7,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3370174,5732129,68.0,45.0,4,2,0,1,2,9.0,9.0,345.4078,-0.023380,367.06,0.042169,7,False
3370323,5732370,81.0,45.0,4,4,0,5,0,20.0,10.0,234.6113,-0.040569,255.09,0.066399,7,False
3370329,5732373,58.0,45.0,4,7,0,3,0,30.0,8.0,174.1869,-0.074677,189.35,0.061103,7,True
3370407,5732452,89.0,45.0,4,1,0,2,0,10.0,23.0,238.8796,-0.060040,259.78,0.064402,7,True


## Section 15 — Linear Regression: Policy Premium (5-Fold OOF)

### `ModelLinearRegressor`
Plain **sklearn `LinearRegression`** with no hyperparameters — serves as the interpretable baseline for premium prediction.

### `cross_validate_linear_regression`
- 5-fold KFold with no tuning overhead
- Reports MAE, RMSE, R² per fold and overall OOF metrics
- Stores OOF predictions as `linear_reg_oof_prediction` → renamed to `Y_hat`

**Dataset:** `df_middle_reg_linear` — same filtering thresholds as other datasets, with label-encoded categoricals.  
`U = (upcoming_premium − Y) / Y` is computed after column renaming.

**Exported to:** `df_exp_financial_loss_linear_black_box.csv`  
Schema: `id, features, Y (premium), U, Y_hat (predicted premium)`

In [396]:
df_middle_reg_linear = filter_middle_advanced(
    df_sub,
    numeric_filters={
        'X_age': {'multiplier': 1.5, 'method': 'iqr'},
        'X_policy_premium': {'multiplier': 2.0, 'method': 'iqr'},
        'X_upcoming_premium':{'multiplier':2.0,'method':'iqr'},
        'X_vehicle_age': {'method': 'percentile', 'lower_q': 0.05, 'upper_q': 0.95},
        'X_cur_renewal_perc':{'method': 'percentile', 'lower_q': 0.05, 'upper_q': 0.95}
    },
    categorical_filters={
        'X_distr_channel': {'max_categories':12},
        'X_risk_code': {'max_categories': 6},
        'X_vehicle_type': {'max_categories': 8}  # explicit list
    },
    keep_id_cols=['encoded_policy_number']
)

X_age: [8.50, 100.50]
X_policy_premium: [-14.06, 444.14]
X_upcoming_premium: [-15.07, 480.13]
X_vehicle_age: [4.00, 31.00]
X_cur_renewal_perc: [-0.02, 0.11]
X_distr_channel: Kept 12 categories
X_risk_code: Kept 6 categories
X_vehicle_type: Kept 8 categories

Total: 463400 → 194373 rows


In [397]:
from sklearn.preprocessing import LabelEncoder
for cat in cat_cols:
    le = LabelEncoder()
    le.fit(df_middle_reg_linear[cat])
    df_middle_reg_linear[cat] = le.transform(df_middle_reg_linear[cat])
    

In [398]:
model_features_reg

['X_age',
 'X_bonus_malus_rating',
 'X_distr_channel',
 'X_vehicle_type',
 'X_ttm_claims',
 'X_policy_count',
 'X_risk_code',
 'X_vehicle_age',
 'X_policy_tenure']

In [400]:
# Regression setup (Linear Regression only)
X_reg = df_middle_reg_linear.loc[:,model_features_reg]
y_reg = df_middle_reg_linear['X_policy_premium']

linear_results = cross_validate_linear_regression(
    X_reg, y_reg, n_splits=5
)

# Save OOF outputs for downstream analysis
df_middle_reg_linear['linear_reg_oof_prediction'] = linear_results['oof_predictions']


Fold 1/5
LinearRegression -> MAE: 36.4032, RMSE: 51.5802, R2: 0.2525

Fold 2/5
LinearRegression -> MAE: 36.0190, RMSE: 50.7388, R2: 0.2541

Fold 3/5
LinearRegression -> MAE: 36.0655, RMSE: 50.9690, R2: 0.2527

Fold 4/5
LinearRegression -> MAE: 36.0176, RMSE: 50.6448, R2: 0.2541

Fold 5/5
LinearRegression -> MAE: 36.0272, RMSE: 50.6479, R2: 0.2555

Overall OOF Metrics
LinearRegression OOF -> MAE: 36.1065, RMSE: 50.9174, R2: 0.2538


In [406]:
df_middle_reg_linear.columns = ['id','X_age', 'X_bonus_malus_rating', 'X_distr_channel',
       'X_vehicle_type', 'X_ttm_claims', 'X_policy_count', 'X_risk_code',
       'X_vehicle_age', 'X_policy_tenure', 'Y', 'X_prev_renewal_perc',
       'X_upcoming_premium', 'X_cur_renewal_perc', 'X_year', '1-Z','Y_hat']

In [409]:
df_middle_reg_linear['U'] = (df_middle_reg_linear['X_upcoming_premium'] - df_middle_reg_linear['Y']) / df_middle_reg_linear['Y']

In [411]:
df_middle_reg_linear.loc[:,['id', 'X_age','X_bonus_malus_rating','X_distr_channel','X_vehicle_type','X_ttm_claims','X_policy_count','X_risk_code','X_vehicle_age','X_policy_tenure',
                     'Y','U','Y_hat']].to_csv('df_exp_financial_loss_linear_black_box.csv',sep=';')

In [412]:
df_middle.columns

Index(['id', 'X_age', 'X_bonus_malus_rating', 'X_distr_channel',
       'X_vehicle_type', 'X_ttm_claims', 'X_policy_count', 'X_risk_code',
       'X_vehicle_age', 'X_policy_tenure', 'Y', 'X_prev_renewal_perc',
       'X_upcoming_premium', 'X_cur_renewal_perc', 'X_year', '1-Z', 'U',
       'fold_number', 'Y_hat', 'Y_residual', 'Y_absolute_error'],
      dtype='object')

## Section 16 — Model Serialisation

Trained models are persisted as **pickle files** in the `artifacts/` directory for reuse without retraining.

### Saved models

| File | Model | Task |
|---|---|---|
| `artifacts/xgb_classifier_churn.pkl` | XGBoost classifier | Churn probability |
| `artifacts/xgb_regressor_policy_premium.pkl` | XGBoost regressor | Policy premium |
| `artifacts/glm_logistic_churn.pkl` | Logistic Regression (GLM) | Churn probability |
| `artifacts/linear_regression_policy_premium.pkl` | Linear Regression | Policy premium |

Models are loaded and retrained only if not already present in the kernel namespace (`globals()` check), enabling fast re-runs of downstream cells without full retraining.

> **Note:** Feature order must be preserved when loading models for inference.  
> Use `model.feature_names_in_` (available on sklearn estimators) to validate column alignment.

In [413]:
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split


ARTIFACTS_DIR = Path("artifacts")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

CLS_PICKLE_PATH = ARTIFACTS_DIR / "xgb_classifier.pkl"
REG_PICKLE_PATH = ARTIFACTS_DIR / "xgb_regressor.pkl"


def _is_xgb_fitted(model) -> bool:
    try:
        _ = model.get_booster()
        return True
    except Exception:
        return False


# ---------------------------
# 1) Build / retrieve classifier
# ---------------------------
if "xgb_classifier_model" in globals() and isinstance(globals()["xgb_classifier_model"], xgb.XGBClassifier):
    xgb_classifier_model = globals()["xgb_classifier_model"]
else:
    X_cls = df_middle_class.loc[:, model_features_class].copy()
    y_cls = df_middle_class["1-Z"].astype(int).copy()

    Xc_train, Xc_val, yc_train, yc_val = train_test_split(
        X_cls,
        y_cls,
        test_size=0.2,
        random_state=123,
        stratify=y_cls,
    )

    cls_params = {
        "random_state": 123,
        "n_estimators": 250,
        "max_depth": 4,
        "learning_rate": 0.05,
        "subsample": 0.9,
        "colsample_bytree": 0.9,
        "eval_metric": "logloss",
        "enable_categorical": True,
        "early_stopping_rounds": 20,
    }
    xgb_classifier_model = xgb.XGBClassifier(**cls_params)
    xgb_classifier_model.fit(Xc_train, yc_train, eval_set=[(Xc_val, yc_val)], verbose=0)


# ---------------------------
# 2) Build / retrieve regressor
# ---------------------------
if "xgb_model" in globals() and isinstance(globals()["xgb_model"], xgb.XGBRegressor):
    xgb_regressor_model = globals()["xgb_model"]
else:
    xgb_regressor_model = None

# If the existing regressor is not fitted, fit one now.
if xgb_regressor_model is None or not _is_xgb_fitted(xgb_regressor_model):
    X_reg = df_middle.loc[:, model_features_reg].copy()
    y_reg_local = df_middle["Y"].copy()

    Xr_train, Xr_val, yr_train, yr_val = train_test_split(
        X_reg,
        y_reg_local,
        test_size=0.2,
        random_state=123,
    )

    reg_params = {
        "random_state": 123,
        "n_estimators": 300,
        "max_depth": 4,
        "learning_rate": 0.05,
        "subsample": 0.9,
        "colsample_bytree": 0.9,
        "objective": "reg:squarederror",
        "enable_categorical": True,
        "early_stopping_rounds": 20,
    }
    xgb_regressor_model = xgb.XGBRegressor(**reg_params)
    xgb_regressor_model.fit(Xr_train, yr_train, eval_set=[(Xr_val, yr_val)], verbose=0)


# ---------------------------
# 3) Save to pickle
# ---------------------------
with open(CLS_PICKLE_PATH, "wb") as f:
    pickle.dump(xgb_classifier_model, f)

with open(REG_PICKLE_PATH, "wb") as f:
    pickle.dump(xgb_regressor_model, f)

print(f"Saved classifier to: {CLS_PICKLE_PATH.resolve()}")
print(f"Saved regressor to:  {REG_PICKLE_PATH.resolve()}")

Saved classifier to: C:\Users\malosett\OneDrive - Assicurazioni Generali S.p.A\coding\pricing_optim\artifacts\xgb_classifier.pkl
Saved regressor to:  C:\Users\malosett\OneDrive - Assicurazioni Generali S.p.A\coding\pricing_optim\artifacts\xgb_regressor.pkl


In [416]:
model_features_class

['X_age',
 'X_bonus_malus_rating',
 'X_distr_channel',
 'X_vehicle_type',
 'X_ttm_claims',
 'X_policy_count',
 'X_risk_code',
 'X_vehicle_age',
 'X_policy_tenure',
 'X_policy_premium',
 'U']

In [417]:
model_features_reg

['X_age',
 'X_bonus_malus_rating',
 'X_distr_channel',
 'X_vehicle_type',
 'X_ttm_claims',
 'X_policy_count',
 'X_risk_code',
 'X_vehicle_age',
 'X_policy_tenure']

## Section 17 — Single-Record Inference (Smoke Test)

Loads the serialised XGBoost classifier and regressor from `artifacts/` and runs a single-record prediction to verify the pickle round-trip is correct.

**Checks performed:**
- Classifier: outputs churn probability and hard class label
- Regressor: outputs predicted policy premium
- Both: column alignment against `feature_names_in_` to catch any schema mismatch

This cell is intended as a **quick sanity check** before handing off models to production or the optimisation module.

## Section 18 — Save All Four Models

Final cell: trains and serialises all four models in a single block:
1. **XGBClassifier** — churn (on `df_middle_class`)  
2. **XGBRegressor** — premium (on `df_middle`)  
3. **GLM logistic** — churn (on `df_middle_glm`); trained with `ModelGLM` at `C=1.0`  
4. **LinearRegression** — premium (on `df_middle_reg_linear`)  

Each model is written to `artifacts/` independently so individual files can be updated without replacing the full set.

In [414]:
import pickle
from pathlib import Path

import pandas as pd


ARTIFACTS_DIR = Path("artifacts")
CLS_PICKLE_PATH = ARTIFACTS_DIR / "xgb_classifier.pkl"
REG_PICKLE_PATH = ARTIFACTS_DIR / "xgb_regressor.pkl"


# 1) Load models from pickle
with open(CLS_PICKLE_PATH, "rb") as f:
    loaded_xgb_classifier = pickle.load(f)

with open(REG_PICKLE_PATH, "rb") as f:
    loaded_xgb_regressor = pickle.load(f)


# 2) Build a single-record input for each model
single_cls_record = df_middle_class.loc[:, model_features_class].iloc[[0]].copy()
single_reg_record = df_middle.loc[:, model_features_reg].iloc[[0]].copy()

# Align columns in case of any ordering mismatch
if hasattr(loaded_xgb_classifier, "feature_names_in_"):
    single_cls_record = single_cls_record.reindex(columns=list(loaded_xgb_classifier.feature_names_in_))
if hasattr(loaded_xgb_regressor, "feature_names_in_"):
    single_reg_record = single_reg_record.reindex(columns=list(loaded_xgb_regressor.feature_names_in_))


# 3) Inference on one record
cls_prob = float(loaded_xgb_classifier.predict_proba(single_cls_record)[:, 1][0])
cls_pred = int(loaded_xgb_classifier.predict(single_cls_record)[0])
reg_pred = float(loaded_xgb_regressor.predict(single_reg_record)[0])

print("Single-record classifier inference:")
print(f"  Predicted churn probability: {cls_prob:.6f}")
print(f"  Predicted churn class:       {cls_pred}")

print("\nSingle-record regressor inference:")
print(f"  Predicted premium:           {reg_pred:.6f}")

print("\nInput record used for classifier:")
print(single_cls_record)

print("\nInput record used for regressor:")
print(single_reg_record)

Single-record classifier inference:
  Predicted churn probability: 0.157302
  Predicted churn class:       0

Single-record regressor inference:
  Predicted premium:           201.582108

Input record used for classifier:
   X_age  X_bonus_malus_rating  X_distr_channel  X_vehicle_type  X_ttm_claims  \
7   93.0                  45.0                5               7             0   

   X_policy_count  X_risk_code  X_vehicle_age  X_policy_tenure  \
7               1            0           24.0             24.0   

   X_policy_premium         U  
7          175.7725  1.123099  

Input record used for regressor:
   X_age  X_bonus_malus_rating  X_distr_channel  X_vehicle_type  X_ttm_claims  \
7   93.0                  45.0                5               7             0   

   X_policy_count  X_risk_code  X_vehicle_age  X_policy_tenure  
7               1            0           24.0             24.0  


In [424]:
df_middle_reg.columns

Index(['encoded_policy_number', 'X_age', 'X_bonus_malus_rating',
       'X_distr_channel', 'X_vehicle_type', 'X_ttm_claims', 'X_policy_count',
       'X_risk_code', 'X_vehicle_age', 'X_policy_tenure', 'X_policy_premium',
       'X_prev_renewal_perc', 'X_upcoming_premium', 'X_cur_renewal_perc',
       'year', 'is_churn', 'U'],
      dtype='object')

In [425]:
# Save all requested models as pickle:
# 1) XGBClassifier (churn)
# 2) XGBRegressor (policy_premium)
# 3) GLM logistic model (churn)
# 4) Linear model (policy_premium)

import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split


ARTIFACTS_DIR = Path("artifacts")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

PATH_XGB_CLASSIFIER = ARTIFACTS_DIR / "xgb_classifier_churn.pkl"
PATH_XGB_REGRESSOR = ARTIFACTS_DIR / "xgb_regressor_policy_premium.pkl"
PATH_GLM_LOGISTIC = ARTIFACTS_DIR / "glm_logistic_churn.pkl"
PATH_LINEAR_MODEL = ARTIFACTS_DIR / "linear_regression_policy_premium.pkl"


def _to_xgb_compatible(df: pd.DataFrame) -> pd.DataFrame:
    df2 = df.copy()
    obj_cols = df2.select_dtypes(include=["object"]).columns
    for col in obj_cols:
        df2[col] = df2[col].astype("category")
    return df2


def _is_fitted_xgb(model) -> bool:
    try:
        _ = model.get_booster()
        return True
    except Exception:
        return False


# ----------------------------
# XGB Classifier (churn)
# ----------------------------
X_cls = df_middle_class.loc[:, model_features_class].copy()
y_cls = df_middle_class["1-Z"].astype(int).copy()
X_cls = _to_xgb_compatible(X_cls)

if "xgb_classifier_model" in globals() and isinstance(globals()["xgb_classifier_model"], xgb.XGBClassifier) and _is_fitted_xgb(globals()["xgb_classifier_model"]):
    xgb_classifier_model = globals()["xgb_classifier_model"]
else:
    Xc_train, Xc_val, yc_train, yc_val = train_test_split(
        X_cls, y_cls, test_size=0.2, random_state=123, stratify=y_cls
    )
    xgb_classifier_model = xgb.XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.9,
        colsample_bytree=0.9,
        eval_metric="logloss",
        random_state=123,
        enable_categorical=True,
        early_stopping_rounds=20,
    )
    xgb_classifier_model.fit(Xc_train, yc_train, eval_set=[(Xc_val, yc_val)], verbose=0)


# ----------------------------
# XGB Regressor (policy_premium)
# ----------------------------
X_reg = df_middle.loc[:, model_features_reg].copy()
y_reg_local = df_middle["Y"].copy()
X_reg = _to_xgb_compatible(X_reg)

if "xgb_model" in globals() and isinstance(globals()["xgb_model"], xgb.XGBRegressor) and _is_fitted_xgb(globals()["xgb_model"]):
    xgb_regressor_model = globals()["xgb_model"]
else:
    Xr_train, Xr_val, yr_train, yr_val = train_test_split(
        X_reg, y_reg_local, test_size=0.2, random_state=123
    )
    xgb_regressor_model = xgb.XGBRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.9,
        colsample_bytree=0.9,
        objective="reg:squarederror",
        random_state=123,
        enable_categorical=True,
        early_stopping_rounds=20,
    )
    xgb_regressor_model.fit(Xr_train, yr_train, eval_set=[(Xr_val, yr_val)], verbose=0)


# ----------------------------
# GLM logistic model (churn)
# ----------------------------
X_glm = df_middle_glm.loc[:, model_features_class].copy()
y_glm = df_middle_glm["1-Z"].astype(int).copy()

Xg_train, Xg_val, yg_train, yg_val = train_test_split(
    X_glm, y_glm, test_size=0.2, random_state=123, stratify=y_glm
)

glm_model_obj = ModelGLM()
glm_logistic_model = glm_model_obj.train(
    Xg_train,
    yg_train,
    Xg_val,
    yg_val,
    {"C": 1.0},
)


# ----------------------------
# Linear model (policy_premium)
# ----------------------------
if "df_middle_reg" in globals():
    X_lin = df_middle_reg_linear.loc[:, model_features_reg].copy()
    y_lin = df_middle_reg_linear["Y"].copy()
else:
    X_lin = df_middle.loc[:, model_features_reg].copy()
    y_lin = df_middle["Y"].copy()

linear_model = LinearRegression()
linear_model.fit(X_lin, y_lin)


# ----------------------------
# Save all models
# ----------------------------
with open(PATH_XGB_CLASSIFIER, "wb") as f:
    pickle.dump(xgb_classifier_model, f)

with open(PATH_XGB_REGRESSOR, "wb") as f:
    pickle.dump(xgb_regressor_model, f)

with open(PATH_GLM_LOGISTIC, "wb") as f:
    pickle.dump(glm_logistic_model, f)

with open(PATH_LINEAR_MODEL, "wb") as f:
    pickle.dump(linear_model, f)

print(f"Saved: {PATH_XGB_CLASSIFIER.resolve()}")
print(f"Saved: {PATH_XGB_REGRESSOR.resolve()}")
print(f"Saved: {PATH_GLM_LOGISTIC.resolve()}")
print(f"Saved: {PATH_LINEAR_MODEL.resolve()}")

Saved: C:\Users\malosett\OneDrive - Assicurazioni Generali S.p.A\coding\pricing_optim\artifacts\xgb_classifier_churn.pkl
Saved: C:\Users\malosett\OneDrive - Assicurazioni Generali S.p.A\coding\pricing_optim\artifacts\xgb_regressor_policy_premium.pkl
Saved: C:\Users\malosett\OneDrive - Assicurazioni Generali S.p.A\coding\pricing_optim\artifacts\glm_logistic_churn.pkl
Saved: C:\Users\malosett\OneDrive - Assicurazioni Generali S.p.A\coding\pricing_optim\artifacts\linear_regression_policy_premium.pkl
